# Herzprung-Russell Diagram

In [2]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


OUTPUT_FORMAT = "mp4"  # "webm" | "gif" | "mp4"
FPS = 24
TOTAL_FRAMES = 180

ANIMATIONS_DIR = Path("media-site/animations")
ANIMATION_NAME = "telemetry_hr_diagram_v2"

CANVAS_SIZE = (1200, 680)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)


def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 4):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t: float) -> float:
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3 - 2 * t)


def lerp(a, b, t):
    return a + (b - a) * t


def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)
    W, H = img.size

    pad = 28
    cut = 48

    pts = [
        (pad + cut, pad),
        (W - pad - 260, pad),
        (W - pad - 230, pad + 18),
        (W - pad, pad + 18),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)

    d.rectangle(
        [pad + 18, pad + 38, W - pad - 18, H - pad - 44],
        outline=(*CYAN2, 38),
        width=1,
    )

    d.text(
        (pad + 24, pad + 8),
        "TELEMETRY // HERTZSPRUNG-RUSSELL DIAGRAM",
        font=font,
        fill=(*CYAN2, 225),
    )


def draw_grid_background(img: Image.Image):
    d = ImageDraw.Draw(img)
    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 78, x, H - 64], fill=(*CYAN, 10), width=1)

    for y in range(100, H - 70, 60):
        d.line([70, y, W - 70, y], fill=(*CYAN, 9), width=1)


def hr_box():
    return 82, 128, 835, 525


def hr_map(log_temp: float, log_lum: float) -> tuple[float, float]:
    x0, y0, x1, y1 = hr_box()

    t_min, t_max = 3.35, 4.65
    l_min, l_max = -4.8, 6.6

    x = x0 + (t_max - log_temp) / (t_max - t_min) * (x1 - x0)
    y = y1 - (log_lum - l_min) / (l_max - l_min) * (y1 - y0)

    return x, y


def spectral_color(log_temp: float) -> tuple[int, int, int]:
    if log_temp > 4.45:
        return (110, 170, 255)
    if log_temp > 4.20:
        return (80, 220, 255)
    if log_temp > 3.98:
        return (180, 255, 255)
    if log_temp > 3.82:
        return (130, 255, 140)
    if log_temp > 3.68:
        return (255, 240, 60)
    if log_temp > 3.52:
        return (255, 150, 45)
    return (255, 60, 45)


def generate_hr_catalog(seed: int = 31415):
    rng = np.random.default_rng(seed)
    stars = []

    for _ in range(1450):
        u = rng.beta(1.25, 1.15)
        log_t = lerp(4.52, 3.42, u) + rng.normal(0, 0.045)
        log_l = lerp(5.3, -3.2, u) + rng.normal(0, 0.42)
        r = rng.uniform(0.9, 2.2)
        stars.append(("main", log_t, log_l, r))

    for _ in range(420):
        u = rng.random()
        log_t = lerp(3.72, 3.45, u) + rng.normal(0, 0.035)
        log_l = lerp(1.0, 4.1, u) + rng.normal(0, 0.35)
        r = rng.uniform(1.3, 2.8)
        stars.append(("giant", log_t, log_l, r))

    for _ in range(220):
        log_t = rng.uniform(3.55, 4.15)
        log_l = rng.normal(5.7, 0.28)
        r = rng.uniform(1.5, 3.2)
        stars.append(("supergiant", log_t, log_l, r))

    for _ in range(360):
        u = rng.random()
        log_t = lerp(4.42, 3.78, u) + rng.normal(0, 0.04)
        log_l = lerp(-1.0, -3.8, u) + rng.normal(0, 0.28)
        r = rng.uniform(0.8, 1.8)
        stars.append(("wd", log_t, log_l, r))

    rng.shuffle(stars)
    return stars


STAR_CATALOG = generate_hr_catalog()


def draw_hr_axes(layer: Image.Image, font_axis, font_tick):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = hr_box()

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 120))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 110), width=1)

    temp_ticks = [
        (4.60, "40000 K"),
        (4.30, "20000 K"),
        (4.00, "10000 K"),
        (3.70, "5000 K"),
        (3.40, "2500 K"),
    ]

    for log_t, label in temp_ticks:
        x, _ = hr_map(log_t, -4.8)
        d.line([x, y0, x, y1], fill=(*CYAN, 26), width=1)
        d.text((x - 36, y1 + 12), label, font=font_tick, fill=(*CYAN2, 185))

    lum_ticks = [
        (6, "10^6"),
        (4, "10^4"),
        (2, "10^2"),
        (0, "1"),
        (-2, "10^-2"),
        (-4, "10^-4"),
    ]

    for log_l, label in lum_ticks:
        _, y = hr_map(3.35, log_l)
        d.line([x0, y, x1, y], fill=(*CYAN, 24), width=1)
        d.text((x0 - 72, y - 9), label, font=font_tick, fill=(*CYAN2, 185))

    d.text(
        (x0 + 8, y0 - 30),
        "Luminosity  L / Lsun",
        font=font_axis,
        fill=(*CYAN2, 215),
    )

    d.text(
        (x0 + 285, y1 + 44),
        "Temperature  T_eff, K",
        font=font_axis,
        fill=(*CYAN2, 215),
    )


def draw_region_labels(layer: Image.Image, font_mid, font_small):
    d = ImageDraw.Draw(layer)

    d.text((236, 314), "Main Sequence", font=font_mid, fill=(*CYAN2, 185))
    d.text((650, 142), "Supergiants", font=font_mid, fill=(*CYAN2, 195))
    d.text((672, 292), "Giants", font=font_mid, fill=(*CYAN2, 190))
    d.text((286, 445), "White Dwarfs", font=font_mid, fill=(*CYAN2, 190))

    d.line([550, 306, 585, 280], fill=(*CYAN2, 90), width=1)
    d.line([632, 170, 682, 156], fill=(*CYAN2, 90), width=1)
    d.line([620, 314, 668, 314], fill=(*CYAN2, 90), width=1)
    d.line([395, 440, 446, 420], fill=(*CYAN2, 90), width=1)


def draw_star_cloud(layer: Image.Image, progress: float, phase: float):
    d = ImageDraw.Draw(layer)

    visible = int(len(STAR_CATALOG) * smoothstep(progress))

    for idx, (_kind, log_t, log_l, radius) in enumerate(STAR_CATALOG[:visible]):
        x, y = hr_map(log_t, log_l)
        color = spectral_color(log_t)

        shimmer = 0.75 + 0.25 * np.sin(phase * 2.0 + idx * 0.19) ** 2
        alpha = int(90 * shimmer)

        if idx > visible - 70:
            alpha = int(40 + 140 * (idx - max(0, visible - 70)) / 70)

        d.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            fill=(*color, alpha),
        )

    rng = np.random.default_rng(123)
    sample_count = int(55 * smoothstep(progress))

    for j in range(sample_count):
        _kind, log_t, log_l, _r = STAR_CATALOG[j * 17 % len(STAR_CATALOG)]
        x, y = hr_map(log_t, log_l)
        color = spectral_color(log_t)
        rr = 3.0 + 2.5 * rng.random()

        d.ellipse(
            [x - rr, y - rr, x + rr, y + rr],
            fill=(*color, 205),
        )


def draw_target(layer: Image.Image, phase: float, font_small):
    d = ImageDraw.Draw(layer)

    log_t = 3.768 + 0.015 * np.sin(phase)
    log_l = 0.00 + 0.10 * np.sin(phase * 2)

    x, y = hr_map(log_t, log_l)

    pulse = 0.55 + 0.45 * np.sin(phase * 4) ** 2

    d.ellipse([x - 18, y - 18, x + 18, y + 18], outline=(*GREEN, int(130 + 100 * pulse)), width=2)
    d.ellipse([x - 7, y - 7, x + 7, y + 7], outline=(*YELLOW, 230), width=2)
    d.line([x - 28, y, x + 28, y], fill=(*GREEN, 160), width=1)
    d.line([x, y - 28, x, y + 28], fill=(*GREEN, 160), width=1)

    d.text((x + 20, y - 34), "SUN", font=font_small, fill=(*GREEN, 210))

    return log_t, log_l


def draw_spectral_bar(layer: Image.Image, font_mid):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = 110, 594, 835, 624

    classes = [
        ("O", (80, 35, 255)),
        ("B", (30, 80, 255)),
        ("A", (80, 190, 255)),
        ("F", (220, 245, 235)),
        ("G", (255, 220, 50)),
        ("K", (255, 130, 35)),
        ("M", (255, 40, 35)),
    ]

    seg_w = (x1 - x0) / len(classes)

    for i, (label, color) in enumerate(classes):
        xa = x0 + i * seg_w
        xb = x0 + (i + 1) * seg_w

        d.rectangle([xa, y0, xb, y1], fill=(*color, 150))
        d.rectangle([xa, y0, xb, y1], outline=(*CYAN2, 45), width=1)

        bbox = d.textbbox((0, 0), label, font=font_mid)
        tw = bbox[2] - bbox[0]
        d.text((xa + seg_w / 2 - tw / 2, y0 + 3), label, font=font_mid, fill=(*WHITE, 220))


def draw_telemetry_box(layer: Image.Image, box: tuple[int, int, int, int], rows: list[tuple[str, str]], font):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    cut = 18
    pts = [
        (x0 + cut, y0),
        (x1, y0),
        (x1, y1 - cut),
        (x1 - cut, y1),
        (x0, y1),
        (x0, y0 + cut),
    ]

    d.polygon(pts, fill=(*BG_DARK, 160))
    d.line(pts + [pts[0]], fill=(*CYAN, 105), width=1)

    y = y0 + 22

    for key, value in rows:
        d.text((x0 + 22, y), key, font=font, fill=(*CYAN2, 170))
        d.text((x0 + 120, y), value, font=font, fill=(*GREEN, 225))
        y += 32


def draw_progress_panel(layer: Image.Image, box: tuple[int, int, int, int], progress: float, font):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    cut = 18
    pts = [
        (x0 + cut, y0),
        (x1, y0),
        (x1, y1 - cut),
        (x1 - cut, y1),
        (x0, y1),
        (x0, y0 + cut),
    ]

    d.polygon(pts, fill=(*BG_DARK, 160))
    d.line(pts + [pts[0]], fill=(*CYAN, 105), width=1)

    d.text((x0 + 22, y0 + 20), "POPULATION PROGRESS", font=font, fill=(*CYAN2, 190))

    bx0, by0 = x0 + 22, y0 + 64
    bx1, by1 = x1 - 22, y0 + 84

    segs = 34
    filled = int(segs * smoothstep(progress))

    for i in range(segs):
        xa = bx0 + i * ((bx1 - bx0) / segs)
        xb = xa + ((bx1 - bx0) / segs) - 2
        active = i < filled
        col = GREEN if active else CYAN
        alpha = 190 if active else 45
        d.rectangle([xa, by0, xb, by1], fill=(*col, alpha))

    count = int(len(STAR_CATALOG) * smoothstep(progress))
    d.text((x0 + 22, y0 + 106), f"Stars plotted     {count:04d}", font=font, fill=(*GREEN, 190))


def draw_legend(layer: Image.Image, box: tuple[int, int, int, int], font):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    cut = 18
    pts = [
        (x0 + cut, y0),
        (x1, y0),
        (x1, y1 - cut),
        (x1 - cut, y1),
        (x0, y1),
        (x0, y0 + cut),
    ]

    d.polygon(pts, fill=(*BG_DARK, 160))
    d.line(pts + [pts[0]], fill=(*CYAN, 105), width=1)

    d.text((x0 + 18, y0 + 14), "LEGEND", font=font, fill=(*CYAN2, 190))

    items = [
        ("Main Sequence", CYAN2),
        ("Giants", ORANGE),
        ("Supergiants", YELLOW),
        ("White Dwarfs", PURPLE),
    ]

    yy = y0 + 44

    for label, color in items:
        d.ellipse(
            [x0 + 20, yy + 3, x0 + 32, yy + 15],
            fill=(*color, 210),
        )
        d.text(
            (x0 + 46, yy),
            label,
            font=font,
            fill=(*CYAN2, 180),
        )
        yy += 23


def draw_hr_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)

    font_big = load_font(22)
    font_mid = load_font(19)
    font_small = load_font(15)
    font_tick = load_font(15)
    font_axis = load_font(17)
    font_legend = load_font(13)

    phase = 2 * np.pi * i / TOTAL_FRAMES
    progress = i / TOTAL_FRAMES

    draw_hud_frame(frame, font_big)
    draw_grid_background(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))

    draw_hr_axes(scene, font_axis, font_tick)
    draw_star_cloud(scene, progress, phase)
    draw_region_labels(scene, font_mid, font_small)

    log_t, log_l = draw_target(scene, phase, font_small)

    draw_spectral_bar(scene, font_mid)

    temp = 10 ** log_t
    lum = 10 ** log_l

    rows = [
        ("T_eff", f"{temp:,.0f} K"),
        ("log L", f"{log_l:+.2f}"),
        ("L/Lsun", f"{lum:.2f}"),
        ("class", "G2 V"),
        ("stage", "Main Sequence"),
        ("mass", "1.05 Msun"),
        ("radius", "1.01 Rsun"),
    ]

    draw_telemetry_box(scene, (900, 68, 1155, 307), rows, font_small)
    draw_progress_panel(scene, (900, 324, 1155, 462), progress, font_small)
    draw_legend(scene, (900, 486, 1155, 625), font_legend)

    glow_composite(frame, scene, blur=4)

    return frame


def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "34",
            "-pix_fmt", "yuva420p",
            "-auto-alt-ref", "0",
            "-row-mt", "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "22",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str) -> Path:
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] HR diagram telemetry v2")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")
        frames.append(draw_hr_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(
        f"Created 1 animation(s) as '{output_format}' "
        f"in '{ANIMATIONS_DIR.resolve()}'"
    )


main()

[START] HR diagram telemetry v2
[CONFIG] OUTPUT_FORMAT = mp4
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 180
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/180
[GENERATE] frame 24/180
[GENERATE] frame 48/180
[GENERATE] frame 72/180
[GENERATE] frame 96/180
[GENERATE] frame 120/180
[GENERATE] frame 144/180
[GENERATE] frame 168/180


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] telemetry_hr_diagram_v2
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/telemetry_hr_diagram_v2/telemetry_hr_diagram_v2.mp4

Created 1 animation(s) as 'mp4' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


[mp4 @ 0x13c014a00] Starting second pass: moving the moov atom to the beginning of the file.3x    
[out#0/mp4 @ 0x13c105740] video:363KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.806567%
frame=  180 fps=0.0 q=-1.0 Lsize=     366KiB time=00:00:07.41 bitrate= 403.9kbits/s speed=13.8x    
[libx264 @ 0x13c106030] frame I:1     Avg QP:15.31  size: 40392
[libx264 @ 0x13c106030] frame P:46    Avg QP:18.75  size:  5267
[libx264 @ 0x13c106030] frame B:133   Avg QP:26.77  size:   662
[libx264 @ 0x13c106030] consecutive B-frames:  0.6%  1.1%  5.0% 93.3%
[libx264 @ 0x13c106030] mb I  I16..4: 59.3% 14.4% 26.3%
[libx264 @ 0x13c106030] mb P  I16..4:  2.5%  0.1%  0.2%  P16..4:  7.9%  0.8%  1.3%  0.0%  0.0%    skip:87.1%
[libx264 @ 0x13c106030] mb B  I16..4:  2.2%  0.0%  0.0%  B16..8:  5.4%  0.3%  0.1%  direct: 0.2%  skip:91.8%  L0:31.2% L1:67.1% BI: 1.7%
[libx264 @ 0x13c106030] 8x8 transform intra:3.8% inter:31.0%
[libx264 @ 0x13c106030] coded y,uvDC,uvAC intr

In [29]:
from __future__ import annotations



from pathlib import Path

import shutil

import subprocess

import numpy as np

from PIL import Image, ImageDraw, ImageFilter, ImageFont





OUTPUT_FORMAT = "webm"  # "webm" | "gif" | "mp4"

FPS = 24

TOTAL_FRAMES = 180



ANIMATIONS_DIR = Path("media-site/animations")

ANIMATION_NAME = "stellar_evolution_very_massive_star"



CANVAS_SIZE = (1200, 680)



BG_DARK = (2, 7, 13)

CYAN = (90, 240, 255)

CYAN2 = (180, 255, 255)

GREEN = (90, 255, 170)

YELLOW = (255, 220, 90)

ORANGE = (255, 160, 70)

RED = (255, 80, 110)

PURPLE = (190, 120, 255)

WHITE = (245, 250, 255)

BLACK = (0, 0, 0)





def normalize_output_format(fmt: str) -> str:

    fmt = fmt.lower().strip()

    if fmt not in {"webm", "gif", "mp4"}:

        raise ValueError("OUTPUT_FORMAT must be webm, gif or mp4")

    return fmt





def load_font(size: int):

    for path in [

        "/System/Library/Fonts/Menlo.ttc",

        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",

        "/Library/Fonts/Arial.ttf",

        "DejaVuSansMono.ttf",

    ]:

        try:

            return ImageFont.truetype(path, size)

        except Exception:

            pass

    return ImageFont.load_default()





def make_canvas(output_format: str) -> Image.Image:

    if normalize_output_format(output_format) == "webm":

        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))

    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))





def flatten_to_black(frame: Image.Image) -> Image.Image:

    frame = frame.convert("RGBA")

    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))

    bg.alpha_composite(frame)

    return bg.convert("RGB")





def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 4):

    glow = layer.filter(ImageFilter.GaussianBlur(blur))

    base.alpha_composite(glow)

    base.alpha_composite(layer)





def smoothstep(t: float) -> float:

    t = float(np.clip(t, 0.0, 1.0))

    return t * t * (3 - 2 * t)





def lerp(a, b, t):

    return a + (b - a) * t





def mix_color(c1, c2, t):

    t = float(np.clip(t, 0, 1))

    return tuple(int(lerp(a, b, t)) for a, b in zip(c1, c2))





def draw_hud_frame(img: Image.Image, font):

    d = ImageDraw.Draw(img)

    W, H = img.size



    pad = 28

    cut = 48



    pts = [

        (pad + cut, pad),

        (W - pad - 260, pad),

        (W - pad - 230, pad + 18),

        (W - pad, pad + 18),

        (W - pad, H - pad - cut),

        (W - pad - cut, H - pad),

        (pad, H - pad),

        (pad, pad + cut),

    ]



    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)

    d.rectangle([pad + 18, pad + 38, W - pad - 18, H - pad - 44], outline=(*CYAN2, 38), width=1)



    d.text(

        (pad + 24, pad + 8),

        "TELEMETRY // STELLAR EVOLUTION TRACK // VERY MASSIVE STAR",

        font=font,

        fill=(*CYAN2, 225),

    )





def draw_grid_background(img: Image.Image):

    d = ImageDraw.Draw(img)

    W, H = img.size



    for x in range(80, W - 80, 80):

        d.line([x, 78, x, H - 64], fill=(*CYAN, 10), width=1)



    for y in range(100, H - 70, 60):

        d.line([70, y, W - 70, y], fill=(*CYAN, 9), width=1)





def hr_box():

    return 82, 128, 835, 525





def hr_map(log_temp: float, log_lum: float) -> tuple[float, float]:

    x0, y0, x1, y1 = hr_box()



    t_min, t_max = 3.35, 4.65

    l_min, l_max = -4.8, 6.6



    x = x0 + (t_max - log_temp) / (t_max - t_min) * (x1 - x0)

    y = y1 - (log_lum - l_min) / (l_max - l_min) * (y1 - y0)



    return x, y





def spectral_color(log_temp: float) -> tuple[int, int, int]:

    if log_temp > 4.45:

        return (110, 170, 255)

    if log_temp > 4.20:

        return (80, 220, 255)

    if log_temp > 3.98:

        return (180, 255, 255)

    if log_temp > 3.82:

        return (130, 255, 140)

    if log_temp > 3.68:

        return (255, 240, 60)

    if log_temp > 3.52:

        return (255, 150, 45)

    return (255, 60, 45)





def generate_hr_catalog(seed: int = 31415):

    rng = np.random.default_rng(seed)

    stars = []



    for _ in range(1450):

        u = rng.beta(1.25, 1.15)

        log_t = lerp(4.52, 3.42, u) + rng.normal(0, 0.045)

        log_l = lerp(5.3, -3.2, u) + rng.normal(0, 0.42)

        r = rng.uniform(0.7, 1.7)

        stars.append(("main", log_t, log_l, r))



    for _ in range(420):

        u = rng.random()

        log_t = lerp(3.72, 3.45, u) + rng.normal(0, 0.035)

        log_l = lerp(1.0, 4.1, u) + rng.normal(0, 0.35)

        r = rng.uniform(1.0, 2.2)

        stars.append(("giant", log_t, log_l, r))



    for _ in range(220):

        log_t = rng.uniform(3.55, 4.15)

        log_l = rng.normal(5.7, 0.28)

        r = rng.uniform(1.2, 2.6)

        stars.append(("supergiant", log_t, log_l, r))



    for _ in range(360):

        u = rng.random()

        log_t = lerp(4.42, 3.78, u) + rng.normal(0, 0.04)

        log_l = lerp(-1.0, -3.8, u) + rng.normal(0, 0.28)

        r = rng.uniform(0.6, 1.4)

        stars.append(("wd", log_t, log_l, r))



    rng.shuffle(stars)

    return stars





STAR_CATALOG = generate_hr_catalog()





def draw_hr_axes(layer: Image.Image, font_axis, font_tick):

    d = ImageDraw.Draw(layer)

    x0, y0, x1, y1 = hr_box()



    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 120))

    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 110), width=1)



    temp_ticks = [

        (4.60, "40000 K"),

        (4.30, "20000 K"),

        (4.00, "10000 K"),

        (3.70, "5000 K"),

        (3.40, "2500 K"),

    ]



    for log_t, label in temp_ticks:

        x, _ = hr_map(log_t, -4.8)

        d.line([x, y0, x, y1], fill=(*CYAN, 26), width=1)

        d.text((x - 36, y1 + 12), label, font=font_tick, fill=(*CYAN2, 185))



    lum_ticks = [(6, "10^6"), (4, "10^4"), (2, "10^2"), (0, "1"), (-2, "10^-2"), (-4, "10^-4")]



    for log_l, label in lum_ticks:

        _, y = hr_map(3.35, log_l)

        d.line([x0, y, x1, y], fill=(*CYAN, 24), width=1)

        d.text((x0 - 72, y - 9), label, font=font_tick, fill=(*CYAN2, 185))



    d.text((x0 + 8, y0 - 30), "Luminosity  L / Lsun", font=font_axis, fill=(*CYAN2, 215))

    d.text((x0 + 285, y1 + 44), "Temperature  T_eff, K", font=font_axis, fill=(*CYAN2, 215))





def draw_region_labels(layer: Image.Image, font_mid):

    d = ImageDraw.Draw(layer)



    d.text((236, 314), "Main Sequence", font=font_mid, fill=(*CYAN2, 150))

    d.text((650, 142), "Supergiants", font=font_mid, fill=(*CYAN2, 165))

    d.text((672, 292), "Giants", font=font_mid, fill=(*CYAN2, 150))

    d.text((286, 445), "White Dwarfs", font=font_mid, fill=(*CYAN2, 145))





def draw_star_cloud(layer: Image.Image):

    d = ImageDraw.Draw(layer)



    for idx, (_kind, log_t, log_l, radius) in enumerate(STAR_CATALOG):

        x, y = hr_map(log_t, log_l)

        color = spectral_color(log_t)

        alpha = 38 if idx % 7 else 72



        d.ellipse(

            [x - radius, y - radius, x + radius, y + radius],

            fill=(*color, alpha),

        )





def draw_spectral_bar(layer: Image.Image, font_mid):

    d = ImageDraw.Draw(layer)

    x0, y0, x1, y1 = 110, 594, 835, 624



    classes = [

        ("O", (80, 35, 255)),

        ("B", (30, 80, 255)),

        ("A", (80, 190, 255)),

        ("F", (220, 245, 235)),

        ("G", (255, 220, 50)),

        ("K", (255, 130, 35)),

        ("M", (255, 40, 35)),

    ]



    seg_w = (x1 - x0) / len(classes)



    for i, (label, color) in enumerate(classes):

        xa = x0 + i * seg_w

        xb = x0 + (i + 1) * seg_w



        d.rectangle([xa, y0, xb, y1], fill=(*color, 150))

        d.rectangle([xa, y0, xb, y1], outline=(*CYAN2, 45), width=1)



        bbox = d.textbbox((0, 0), label, font=font_mid)

        tw = bbox[2] - bbox[0]

        d.text((xa + seg_w / 2 - tw / 2, y0 + 3), label, font=font_mid, fill=(*WHITE, 220))





TRACK = [

    {

        "label": "O-STAR MAIN SEQUENCE",

        "t0": 0.0,

        "t1": 6.2,

        "logT0": 4.58,

        "logL0": 5.45,

        "logT1": 4.47,

        "logL1": 5.60,

        "r0": 10,

        "r1": 13,

        "c0": (110, 170, 255),

        "c1": (120, 210, 255),

    },

    {

        "label": "BLUE SUPERGIANT",

        "t0": 6.2,

        "t1": 7.7,

        "logT0": 4.47,

        "logL0": 5.60,

        "logT1": 4.12,

        "logL1": 5.75,

        "r0": 13,

        "r1": 20,

        "c0": (120, 210, 255),

        "c1": (170, 245, 255),

    },

    {

        "label": "RED SUPERGIANT",

        "t0": 7.7,

        "t1": 8.6,

        "logT0": 4.12,

        "logL0": 5.75,

        "logT1": 3.55,

        "logL1": 5.45,

        "r0": 20,

        "r1": 54,

        "c0": (170, 245, 255),

        "c1": (255, 70, 45),

    },

    {

        "label": "CORE COLLAPSE",

        "t0": 8.6,

        "t1": 8.85,

        "logT0": 3.55,

        "logL0": 5.45,

        "logT1": 3.62,

        "logL1": 6.15,

        "r0": 54,

        "r1": 72,

        "c0": (255, 70, 45),

        "c1": (255, 245, 210),

    },

    {

        "label": "BLACK HOLE REMNANT",

        "t0": 8.85,

        "t1": 9.0,

        "logT0": 3.62,

        "logL0": 6.15,

        "logT1": 4.55,

        "logL1": 6.55,

        "r0": 72,

        "r1": 28,

        "c0": (255, 245, 210),

        "c1": (0, 0, 0),

    },

]



TOTAL_MYR = 9.0





def track_state(progress: float):

    age = progress * TOTAL_MYR



    for stage in TRACK:

        if stage["t0"] <= age <= stage["t1"]:

            local = (age - stage["t0"]) / max(1e-9, stage["t1"] - stage["t0"])

            local_s = smoothstep(local)



            log_t = lerp(stage["logT0"], stage["logT1"], local_s)

            log_l = lerp(stage["logL0"], stage["logL1"], local_s)

            radius = lerp(stage["r0"], stage["r1"], local_s)

            color = mix_color(stage["c0"], stage["c1"], local_s)



            return age, stage["label"], log_t, log_l, radius, color, local



    stage = TRACK[-1]

    return TOTAL_MYR, stage["label"], stage["logT1"], stage["logL1"], stage["r1"], stage["c1"], 1.0





def sampled_track_points(progress: float, n: int = 180):

    pts = []

    max_age = progress * TOTAL_MYR



    for age in np.linspace(0, max_age, n):

        p = age / TOTAL_MYR

        _, _label, log_t, log_l, radius, color, _local = track_state(p)

        x, y = hr_map(log_t, log_l)

        pts.append((x, y, radius, color))



    return pts





def draw_evolution_trail(layer: Image.Image, progress: float):

    trail = Image.new("RGBA", layer.size, (0, 0, 0, 0))

    d = ImageDraw.Draw(trail)



    pts = sampled_track_points(progress, n=220)



    if len(pts) < 2:

        return



    for idx in range(1, len(pts)):

        x0, y0, _r0, c0 = pts[idx - 1]

        x1, y1, _r1, c1 = pts[idx]



        alpha = int(40 + 185 * idx / len(pts))

        width = int(5 + 7 * idx / len(pts))

        d.line([x0, y0, x1, y1], fill=(*c1, alpha), width=width)



    glow = trail.filter(ImageFilter.GaussianBlur(6))

    layer.alpha_composite(glow)

    layer.alpha_composite(trail)





def draw_active_star(layer: Image.Image, x: float, y: float, radius: float, color, phase: float, label: str, local_stage: float):

    d = ImageDraw.Draw(layer)



    pulse = 0.82 + 0.18 * np.sin(phase * 5) ** 2



    if label == "RED SUPERGIANT":

        shell_r = radius * (1.35 + 0.08 * np.sin(phase * 6))

        d.ellipse([x - shell_r, y - shell_r, x + shell_r, y + shell_r], fill=(*RED, 38), outline=(*ORANGE, 90), width=2)



    if label == "CORE COLLAPSE":

        shock_r = radius * (1.2 + 1.8 * local_stage)

        d.ellipse([x - shock_r, y - shock_r, x + shock_r, y + shock_r], outline=(*WHITE, int(230 * (1 - local_stage))), width=4)

        d.ellipse([x - shock_r * 1.45, y - shock_r * 1.45, x + shock_r * 1.45, y + shock_r * 1.45], outline=(*YELLOW, int(180 * (1 - local_stage))), width=3)



    if label == "BLACK HOLE REMNANT":

        bh_r = radius * (1.0 + 1.2 * local_stage)

        d.ellipse([x - bh_r * 2.2, y - bh_r * 2.2, x + bh_r * 2.2, y + bh_r * 2.2], fill=(0, 0, 0, int(90 + 140 * local_stage)))

        d.ellipse([x - bh_r, y - bh_r, x + bh_r, y + bh_r], fill=(0, 0, 0, 255), outline=(*PURPLE, 190), width=2)

        d.arc([x - bh_r * 1.6, y - bh_r * 0.7, x + bh_r * 1.6, y + bh_r * 0.7], 0, 360, fill=(*ORANGE, 200), width=3)

        return



    for scale, alpha in [(2.6, 22), (1.8, 48), (1.25, 85)]:

        rr = radius * scale * pulse

        d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(*color, alpha))



    d.ellipse([x - radius, y - radius, x + radius, y + radius], fill=(*color, 235), outline=(*WHITE, 220), width=2)





def draw_stage_timeline(layer: Image.Image, progress: float, font):

    d = ImageDraw.Draw(layer)



    x0, y0, x1, y1 = 110, 594, 835, 624

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 155), outline=(*CYAN, 90), width=1)



    for stage in TRACK:

        xa = x0 + (stage["t0"] / TOTAL_MYR) * (x1 - x0)

        xb = x0 + (stage["t1"] / TOTAL_MYR) * (x1 - x0)

        col = stage["c1"]

        d.rectangle([xa, y0, xb, y1], fill=(*col, 120))

        d.line([xa, y0, xa, y1], fill=(*CYAN2, 75), width=1)



    px = x0 + progress * (x1 - x0)

    d.line([px, y0 - 8, px, y1 + 8], fill=(*WHITE, 230), width=2)

    d.text((x0, y0 - 30), "Evolution timeline, Myr", font=font, fill=(*CYAN2, 185))





def draw_telemetry_box(layer: Image.Image, box: tuple[int, int, int, int], rows: list[tuple[str, str]], font):

    d = ImageDraw.Draw(layer)

    x0, y0, x1, y1 = box



    cut = 18

    pts = [

        (x0 + cut, y0),

        (x1, y0),

        (x1, y1 - cut),

        (x1 - cut, y1),

        (x0, y1),

        (x0, y0 + cut),

    ]



    d.polygon(pts, fill=(*BG_DARK, 165))

    d.line(pts + [pts[0]], fill=(*CYAN, 105), width=1)



    y = y0 + 22

    for key, value in rows:

        d.text((x0 + 22, y), key, font=font, fill=(*CYAN2, 170))

        d.text((x0 + 122, y), value, font=font, fill=(*GREEN, 225))

        y += 32





def draw_event_panel(layer: Image.Image, box: tuple[int, int, int, int], stage_label: str, progress: float, font):

    d = ImageDraw.Draw(layer)

    x0, y0, x1, y1 = box



    cut = 18

    pts = [

        (x0 + cut, y0),

        (x1, y0),

        (x1, y1 - cut),

        (x1 - cut, y1),

        (x0, y1),

        (x0, y0 + cut),

    ]



    d.polygon(pts, fill=(*BG_DARK, 165))

    d.line(pts + [pts[0]], fill=(*CYAN, 105), width=1)



    d.text((x0 + 22, y0 + 18), "FINAL CHANNEL", font=font, fill=(*CYAN2, 190))



    if stage_label == "CORE COLLAPSE":

        status = "SUPERNOVA IMMINENT"

        col = RED

    elif stage_label == "BLACK HOLE REMNANT":

        status = "BLACK HOLE FORMED"

        col = PURPLE

    else:

        status = "CORE BURNING"

        col = GREEN



    d.text((x0 + 22, y0 + 58), status, font=font, fill=(*col, 220))

    d.text((x0 + 22, y0 + 94), "Remnant: BH / NS boundary", font=font, fill=(*CYAN2, 170))





def draw_hr_scene(i: int, output_format: str) -> Image.Image:

    frame = make_canvas(output_format)



    font_big = load_font(20)

    font_mid = load_font(19)

    font_small = load_font(15)

    font_tick = load_font(15)

    font_axis = load_font(17)



    phase = 2 * np.pi * i / TOTAL_FRAMES

    progress = i / (TOTAL_FRAMES - 1)



    age, stage_label, log_t, log_l, radius, color, local_stage = track_state(progress)



    draw_hud_frame(frame, font_big)

    draw_grid_background(frame)



    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))



    draw_hr_axes(scene, font_axis, font_tick)

    draw_star_cloud(scene)

    draw_region_labels(scene, font_mid)



    draw_evolution_trail(scene, progress)



    x, y = hr_map(log_t, log_l)

    draw_active_star(scene, x, y, radius, color, phase, stage_label, local_stage)



    draw_stage_timeline(scene, progress, font_small)



    temp = 10 ** log_t

    lum = 10 ** log_l



    rows = [

        ("mass", "20 Msun"),

        ("age", f"{age:.2f} Myr"),

        ("stage", stage_label),

        ("T_eff", f"{temp:,.0f} K"),

        ("log L", f"{log_l:+.2f}"),

        ("radius", f"{max(1, radius * 12):.0f} Rsun"),

        ("fate", "black hole"),

    ]



    draw_telemetry_box(scene, (900, 68, 1155, 303), rows, font_small)

    draw_event_panel(scene, (900, 330, 1155, 462), stage_label, progress, font_small)



    if stage_label == "BLACK HOLE REMNANT":

        overlay = Image.new("RGBA", frame.size, (0, 0, 0, 0))

        od = ImageDraw.Draw(overlay)

        alpha = int(120 * local_stage)

        od.rectangle([82, 128, 835, 525], fill=(0, 0, 0, alpha))

        scene.alpha_composite(overlay)



    glow_composite(frame, scene, blur=4)

    return frame





def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:

    if shutil.which("ffmpeg") is None:

        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")



    out_path.parent.mkdir(parents=True, exist_ok=True)

    frame_dir = out_path.parent / "_frames"

    frame_dir.mkdir(parents=True, exist_ok=True)



    for idx, frame in enumerate(frames):

        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")



    subprocess.run(

        [

            "ffmpeg", "-y",

            "-framerate", str(fps),

            "-i", str(frame_dir / "frame_%04d.png"),

            "-c:v", "libvpx-vp9",

            "-b:v", "0",

            "-crf", "34",

            "-pix_fmt", "yuva420p",

            "-auto-alt-ref", "0",

            "-row-mt", "1",

            str(out_path),

        ],

        check=True,

    )



    if not keep_frames:

        shutil.rmtree(frame_dir, ignore_errors=True)



    return out_path





def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:

    if shutil.which("ffmpeg") is None:

        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")



    out_path.parent.mkdir(parents=True, exist_ok=True)

    frame_dir = out_path.parent / "_frames"

    frame_dir.mkdir(parents=True, exist_ok=True)



    for idx, frame in enumerate(frames):

        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")



    subprocess.run(

        [

            "ffmpeg", "-y",

            "-framerate", str(fps),

            "-i", str(frame_dir / "frame_%04d.png"),

            "-c:v", "libx264",

            "-crf", "22",

            "-pix_fmt", "yuv420p",

            "-movflags", "+faststart",

            str(out_path),

        ],

        check=True,

    )



    if not keep_frames:

        shutil.rmtree(frame_dir, ignore_errors=True)



    return out_path





def export_gif(frames: list[Image.Image], out_path: Path, fps: int) -> Path:

    out_path.parent.mkdir(parents=True, exist_ok=True)



    gif_frames = [

        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)

        for frame in frames

    ]



    gif_frames[0].save(

        out_path,

        save_all=True,

        append_images=gif_frames[1:],

        duration=int(1000 / fps),

        loop=0,

        disposal=2,

    )



    return out_path





def export_animation(frames: list[Image.Image], output_format: str) -> Path:

    output_format = normalize_output_format(output_format)



    folder = ANIMATIONS_DIR / ANIMATION_NAME

    folder.mkdir(parents=True, exist_ok=True)



    out_path = folder / f"{ANIMATION_NAME}.{output_format}"



    if output_format == "webm":

        return export_webm(frames, out_path, FPS)



    if output_format == "mp4":

        return export_mp4(frames, out_path, FPS)



    return export_gif(frames, out_path, FPS)





def main():

    output_format = normalize_output_format(OUTPUT_FORMAT)



    print("[START] stellar evolution track: very massive star")

    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")

    print(f"[CONFIG] FPS = {FPS}")

    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")

    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")



    frames = []



    for i in range(TOTAL_FRAMES):

        if i % 24 == 0:

            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")

        frames.append(draw_hr_scene(i, output_format))



    saved_path = export_animation(frames, output_format)



    print()

    print(f"[CREATED] {ANIMATION_NAME}")

    print(f"          {saved_path.resolve()}")

    print()

    print(f"Created 1 animation(s) as '{output_format}' in '{ANIMATIONS_DIR.resolve()}'")





main()


[START] stellar evolution track: very massive star
[CONFIG] OUTPUT_FORMAT = gif
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 180
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/180
[GENERATE] frame 24/180
[GENERATE] frame 48/180
[GENERATE] frame 72/180
[GENERATE] frame 96/180
[GENERATE] frame 120/180
[GENERATE] frame 144/180
[GENERATE] frame 168/180

[CREATED] stellar_evolution_very_massive_star
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/stellar_evolution_very_massive_star/stellar_evolution_very_massive_star.gif

Created 1 animation(s) as 'gif' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


In [32]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import textwrap
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


OUTPUT_FORMAT = "webm"  # "webm" | "gif" | "mp4"
FPS = 24

TRACK = [
    {
        "label": "O-STAR MAIN SEQUENCE",
        "t0": 0.0,
        "t1": 6.2,
        "logT0": 4.58,
        "logL0": 5.45,
        "logT1": 4.47,
        "logL1": 5.60,
        "r0": 10,
        "r1": 13,
        "c0": (110, 170, 255),
        "c1": (120, 210, 255),
    },
    {
        "label": "BLUE SUPERGIANT",
        "t0": 6.2,
        "t1": 7.7,
        "logT0": 4.47,
        "logL0": 5.60,
        "logT1": 4.12,
        "logL1": 5.75,
        "r0": 13,
        "r1": 20,
        "c0": (120, 210, 255),
        "c1": (170, 245, 255),
    },
    {
        "label": "RED SUPERGIANT",
        "t0": 7.7,
        "t1": 8.6,
        "logT0": 4.12,
        "logL0": 5.75,
        "logT1": 3.55,
        "logL1": 5.45,
        "r0": 20,
        "r1": 54,
        "c0": (170, 245, 255),
        "c1": (255, 70, 45),
    },
    {
        "label": "CORE COLLAPSE",
        "t0": 8.6,
        "t1": 8.85,
        "logT0": 3.55,
        "logL0": 5.45,
        "logT1": 3.62,
        "logL1": 6.15,
        "r0": 54,
        "r1": 72,
        "c0": (255, 70, 45),
        "c1": (255, 245, 210),
    },
    {
        "label": "BLACK HOLE REMNANT",
        "t0": 8.85,
        "t1": 9.0,
        "logT0": 3.62,
        "logL0": 6.15,
        "logT1": 4.55,
        "logL1": 6.55,
        "r0": 72,
        "r1": 28,
        "c0": (255, 245, 210),
        "c1": (0, 0, 0),
    },
]

ANIMATIONS_DIR = Path("media-site/animations")
ANIMATION_NAME = "stellar_evolution_solar_star"

CANVAS_SIZE = (1200, 720)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)
BLACK = (0, 0, 0)


TIMELINE = [
    # label, visual_start, visual_end, pause
    ("O-STAR MAIN SEQUENCE", 0.00, 0.56, False),
    ("O-STAR MAIN SEQUENCE", 0.56, 0.64, True),

    ("BLUE SUPERGIANT",      0.64, 0.74, False),
    ("BLUE SUPERGIANT",      0.74, 0.80, True),

    ("RED SUPERGIANT",       0.80, 0.91, False),
    ("RED SUPERGIANT",       0.91, 0.96, True),

    ("CORE COLLAPSE",        0.96, 0.988, False),

    ("BLACK HOLE REMNANT",   0.988, 1.0, False),
]

TOTAL_MYR = 9.0

MOVE_FRAMES = 42
HOLD_FRAMES = FPS * 3


def build_visual_timeline(
    move_frames: int = MOVE_FRAMES,
    hold_frames: int = HOLD_FRAMES,
):
    segments = []

    for idx, stage in enumerate(TRACK):
        p0 = stage["t0"] / TOTAL_MYR
        p1 = stage["t1"] / TOTAL_MYR

        segments.append({
            "type": "move",
            "p0": p0,
            "p1": p1,
            "frames": move_frames,
        })

        # hold after every stage, including final black hole stage
        segments.append({
            "type": "hold",
            "p0": p1,
            "p1": p1,
            "frames": hold_frames,
        })

    return segments


VISUAL_TIMELINE = build_visual_timeline()
TOTAL_FRAMES = sum(segment["frames"] for segment in VISUAL_TIMELINE)


def visual_progress_with_stage_holds(frame_index: int) -> float:
    frame_index = frame_index % TOTAL_FRAMES
    cursor = 0

    for segment in VISUAL_TIMELINE:
        start = cursor
        end = cursor + segment["frames"]

        if start <= frame_index < end:
            if segment["type"] == "hold":
                return segment["p0"]

            local = (frame_index - start) / max(1, segment["frames"] - 1)
            local = smoothstep(local)

            return lerp(segment["p0"], segment["p1"], local)

        cursor = end

    return 1.0


def is_visual_hold_frame(frame_index: int) -> bool:
    frame_index = frame_index % TOTAL_FRAMES
    cursor = 0

    for segment in VISUAL_TIMELINE:
        start = cursor
        end = cursor + segment["frames"]

        if start <= frame_index < end:
            return segment["type"] == "hold"

        cursor = end

    return False




def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 4):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t: float) -> float:
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3 - 2 * t)


def lerp(a, b, t):
    return a + (b - a) * t


def mix_color(c1, c2, t):
    t = float(np.clip(t, 0, 1))
    return tuple(int(lerp(a, b, t)) for a, b in zip(c1, c2))


def wrap_lines(text: str, width: int = 16) -> list[str]:
    return textwrap.wrap(text, width=width) or [""]


def draw_wrapped_text(
    d: ImageDraw.ImageDraw,
    xy: tuple[float, float],
    text: str,
    font,
    fill,
    width: int = 16,
    line_h: int = 18,
):
    x, y = xy
    for line in wrap_lines(text, width):
        d.text((x, y), line, font=font, fill=fill)
        y += line_h
    return y


def draw_cut_panel(
    d: ImageDraw.ImageDraw,
    box: tuple[int, int, int, int],
    fill=(*BG_DARK, 165),
    outline=(*CYAN, 105),
    cut: int = 18,
):
    x0, y0, x1, y1 = box

    pts = [
        (x0 + cut, y0),
        (x1, y0),
        (x1, y1 - cut),
        (x1 - cut, y1),
        (x0, y1),
        (x0, y0 + cut),
    ]

    d.polygon(pts, fill=fill)
    d.line(pts + [pts[0]], fill=outline, width=1)


def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)
    W, H = img.size

    pad = 28
    cut = 48

    pts = [
        (pad + cut, pad),
        (W - pad - 260, pad),
        (W - pad - 230, pad + 18),
        (W - pad, pad + 18),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)
    d.rectangle(
        [pad + 18, pad + 38, W - pad - 18, H - pad - 44],
        outline=(*CYAN2, 38),
        width=1,
    )

    d.text(
        (pad + 24, pad + 8),
        "TELEMETRY // STELLAR EVOLUTION TRACK // VERY MASSIVE STAR",
        font=font,
        fill=(*CYAN2, 225),
    )


def draw_grid_background(img: Image.Image):
    d = ImageDraw.Draw(img)
    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 78, x, H - 64], fill=(*CYAN, 10), width=1)

    for y in range(100, H - 70, 60):
        d.line([70, y, W - 70, y], fill=(*CYAN, 9), width=1)


def hr_box():
    return 82, 128, 835, 525


def hr_map(log_temp: float, log_lum: float) -> tuple[float, float]:
    x0, y0, x1, y1 = hr_box()

    t_min, t_max = 3.35, 4.65
    l_min, l_max = -4.8, 6.6

    x = x0 + (t_max - log_temp) / (t_max - t_min) * (x1 - x0)
    y = y1 - (log_lum - l_min) / (l_max - l_min) * (y1 - y0)

    return x, y


def spectral_color(log_temp: float) -> tuple[int, int, int]:
    if log_temp > 4.45:
        return (110, 170, 255)
    if log_temp > 4.20:
        return (80, 220, 255)
    if log_temp > 3.98:
        return (180, 255, 255)
    if log_temp > 3.82:
        return (130, 255, 140)
    if log_temp > 3.68:
        return (255, 240, 60)
    if log_temp > 3.52:
        return (255, 150, 45)
    return (255, 60, 45)


def generate_hr_catalog(seed: int = 31415):
    rng = np.random.default_rng(seed)
    stars = []

    for _ in range(1450):
        u = rng.beta(1.25, 1.15)
        log_t = lerp(4.52, 3.42, u) + rng.normal(0, 0.045)
        log_l = lerp(5.3, -3.2, u) + rng.normal(0, 0.42)
        r = rng.uniform(0.7, 1.7)
        stars.append(("main", log_t, log_l, r))

    for _ in range(420):
        u = rng.random()
        log_t = lerp(3.72, 3.45, u) + rng.normal(0, 0.035)
        log_l = lerp(1.0, 4.1, u) + rng.normal(0, 0.35)
        r = rng.uniform(1.0, 2.2)
        stars.append(("giant", log_t, log_l, r))

    for _ in range(220):
        log_t = rng.uniform(3.55, 4.15)
        log_l = rng.normal(5.7, 0.28)
        r = rng.uniform(1.2, 2.6)
        stars.append(("supergiant", log_t, log_l, r))

    for _ in range(360):
        u = rng.random()
        log_t = lerp(4.42, 3.78, u) + rng.normal(0, 0.04)
        log_l = lerp(-1.0, -3.8, u) + rng.normal(0, 0.28)
        r = rng.uniform(0.6, 1.4)
        stars.append(("wd", log_t, log_l, r))

    rng.shuffle(stars)
    return stars


STAR_CATALOG = generate_hr_catalog()


def draw_hr_axes(layer: Image.Image, font_axis, font_tick):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = hr_box()

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 120))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 110), width=1)

    temp_ticks = [
        (4.60, "40000 K"),
        (4.30, "20000 K"),
        (4.00, "10000 K"),
        (3.70, "5000 K"),
        (3.40, "2500 K"),
    ]

    for log_t, label in temp_ticks:
        x, _ = hr_map(log_t, -4.8)
        d.line([x, y0, x, y1], fill=(*CYAN, 26), width=1)
        d.text((x - 36, y1 + 12), label, font=font_tick, fill=(*CYAN2, 185))

    lum_ticks = [
        (6, "10^6"),
        (4, "10^4"),
        (2, "10^2"),
        (0, "1"),
        (-2, "10^-2"),
        (-4, "10^-4"),
    ]

    for log_l, label in lum_ticks:
        _, y = hr_map(3.35, log_l)
        d.line([x0, y, x1, y], fill=(*CYAN, 24), width=1)
        d.text((x0 - 72, y - 9), label, font=font_tick, fill=(*CYAN2, 185))

    d.text((x0 + 8, y0 - 30), "Luminosity  L / Lsun", font=font_axis, fill=(*CYAN2, 215))
    d.text((x0 + 285, y1 + 44), "Temperature  T_eff, K", font=font_axis, fill=(*CYAN2, 215))


def draw_region_labels(layer: Image.Image, font_mid):
    d = ImageDraw.Draw(layer)

    d.text((236, 314), "Main Sequence", font=font_mid, fill=(*CYAN2, 150))
    d.text((650, 142), "Supergiants", font=font_mid, fill=(*CYAN2, 165))
    d.text((672, 292), "Giants", font=font_mid, fill=(*CYAN2, 150))
    d.text((286, 445), "White Dwarfs", font=font_mid, fill=(*CYAN2, 145))


def draw_star_cloud(layer: Image.Image):
    d = ImageDraw.Draw(layer)

    for idx, (_kind, log_t, log_l, radius) in enumerate(STAR_CATALOG):
        x, y = hr_map(log_t, log_l)
        color = spectral_color(log_t)
        alpha = 38 if idx % 7 else 72

        d.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            fill=(*color, alpha),
        )




def track_state(progress: float):
    age = progress * TOTAL_MYR

    for stage in TRACK:
        if stage["t0"] <= age <= stage["t1"]:
            local = (age - stage["t0"]) / max(1e-9, stage["t1"] - stage["t0"])
            local_s = smoothstep(local)

            log_t = lerp(stage["logT0"], stage["logT1"], local_s)
            log_l = lerp(stage["logL0"], stage["logL1"], local_s)
            # unstable post-planetary-nebula evolution
            if stage["label"] == "POST-AGB CORE":
                wobble = (1.0 - local_s)

                log_t += np.sin(local_s * 18) * 0.045 * wobble
                log_l += np.cos(local_s * 14) * 0.035 * wobble
            
            radius = lerp(stage["r0"], stage["r1"], local_s)
            color = mix_color(stage["c0"], stage["c1"], local_s)

            return age, stage["label"], log_t, log_l, radius, color, local

    stage = TRACK[-1]
    return TOTAL_MYR, stage["label"], stage["logT1"], stage["logL1"], stage["r1"], stage["c1"], 1.0


def sampled_track_points(progress: float, n: int = 180):
    pts = []
    max_age = progress * TOTAL_MYR

    for age in np.linspace(0, max_age, n):
        p = age / TOTAL_MYR
        _, _label, log_t, log_l, radius, color, _local = track_state(p)
        x, y = hr_map(log_t, log_l)
        pts.append((x, y, radius, color))

    return pts


def draw_evolution_trail(layer: Image.Image, progress: float):
    trail = Image.new("RGBA", layer.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(trail)

    pts = sampled_track_points(progress, n=220)

    if len(pts) < 2:
        return

    for idx in range(1, len(pts)):
        x0, y0, _r0, c0 = pts[idx - 1]
        x1, y1, _r1, c1 = pts[idx]

        alpha = int(40 + 185 * idx / len(pts))
        width = int(5 + 7 * idx / len(pts))
        d.line([x0, y0, x1, y1], fill=(*c1, alpha), width=width)

    glow = trail.filter(ImageFilter.GaussianBlur(6))
    layer.alpha_composite(glow)
    layer.alpha_composite(trail)


def draw_active_star(layer: Image.Image, x: float, y: float, radius: float, color, phase: float, label: str, local_stage: float):
    d = ImageDraw.Draw(layer)

    pulse = 0.82 + 0.18 * np.sin(phase * 5) ** 2

    if label == "RED SUPERGIANT":
        shell_r = radius * (1.35 + 0.08 * np.sin(phase * 6))
        d.ellipse([x - shell_r, y - shell_r, x + shell_r, y + shell_r], fill=(*RED, 38), outline=(*ORANGE, 90), width=2)

    if label == "CORE COLLAPSE":
        shock_r = radius * (1.2 + 1.8 * local_stage)
        d.ellipse([x - shock_r, y - shock_r, x + shock_r, y + shock_r], outline=(*WHITE, int(230 * (1 - local_stage))), width=4)
        d.ellipse([x - shock_r * 1.45, y - shock_r * 1.45, x + shock_r * 1.45, y + shock_r * 1.45], outline=(*YELLOW, int(180 * (1 - local_stage))), width=3)

    if label == "BLACK HOLE REMNANT":
        bh_r = radius * (1.0 + 1.2 * local_stage)
        d.ellipse([x - bh_r * 2.2, y - bh_r * 2.2, x + bh_r * 2.2, y + bh_r * 2.2], fill=(0, 0, 0, int(90 + 140 * local_stage)))
        d.ellipse([x - bh_r, y - bh_r, x + bh_r, y + bh_r], fill=(0, 0, 0, 255), outline=(*PURPLE, 190), width=2)
        d.arc([x - bh_r * 1.6, y - bh_r * 0.7, x + bh_r * 1.6, y + bh_r * 0.7], 0, 360, fill=(*ORANGE, 200), width=3)
        return

    for scale, alpha in [(2.6, 22), (1.8, 48), (1.25, 85)]:
        rr = radius * scale * pulse
        d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(*color, alpha))

    d.ellipse([x - radius, y - radius, x + radius, y + radius], fill=(*color, 235), outline=(*WHITE, 220), width=2)


def draw_stage_timeline(layer: Image.Image, progress: float, font):
    d = ImageDraw.Draw(layer)

    x0, y0, x1, y1 = 110, 614, 835, 644
    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 155), outline=(*CYAN, 90), width=1)

    for stage in TRACK:
        xa = x0 + (stage["t0"] / TOTAL_MYR) * (x1 - x0)
        xb = x0 + (stage["t1"] / TOTAL_MYR) * (x1 - x0)
        col = stage["c1"]
        d.rectangle([xa, y0, xb, y1], fill=(*col, 120))
        d.line([xa, y0, xa, y1], fill=(*CYAN2, 75), width=1)

    px = x0 + progress * (x1 - x0)
    d.line([px, y0 - 8, px, y1 + 8], fill=(*WHITE, 230), width=2)

    d.text(
        (x0, y1 + 10),
        "Evolution timeline, Myr",
        font=font,
        fill=(*CYAN2, 185),
    )




def draw_star_metrics_panel(
    layer: Image.Image,
    box: tuple[int, int, int, int],
    rows: list[tuple[str, str]],
    font,
):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 18), "STAR METRICS", font=font, fill=(*CYAN2, 190))

    y = y0 + 56

    for key, value in rows:
        d.text((x0 + 22, y), key, font=font, fill=(*CYAN2, 170))
        d.text((x0 + 122, y), value, font=font, fill=(*GREEN, 225))
        y += 28


def wrap_text(text: str, width: int) -> str:
    words = text.split()
    lines = []
    current = ""

    for word in words:
        test = word if not current else current + " " + word

        if len(test) <= width:
            current = test
        else:
            if current:
                lines.append(current)
            current = word

    if current:
        lines.append(current)

    return "\n".join(lines)


def draw_star_evolution_panel(layer, box, stage_label, progress, font):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 16), "STAR EVOLUTION", font=font, fill=(*CYAN2, 190))

    wrapped_stage = wrap_text(stage_label, 16)

    if "BLACK HOLE" in stage_label:
        remnant = "Black Hole"
        source = "Core Collapse"
    elif "CORE COLLAPSE" in stage_label:
        remnant = "SN / BH"
        source = "Si burning"
    elif "RED SUPERGIANT" in stage_label:
        remnant = "Core Collapse"
        source = "He burning"
    elif "BLUE SUPERGIANT" in stage_label:
        remnant = "Red Supergiant"
        source = "H shell burning"
    else:
        remnant = "Blue Supergiant"
        source = "Hydrogen fusion"

    yy = y0 + 52

    d.text((x0 + 22, yy), "Stage:", font=font, fill=(*CYAN2, 170))
    d.multiline_text(
        (x0 + 98, yy),
        wrapped_stage,
        font=font,
        fill=(*GREEN, 225),
        spacing=3,
    )

    yy += 52

    d.text((x0 + 22, yy), "Source:", font=font, fill=(*CYAN2, 170))
    d.text((x0 + 98, yy), source, font=font, fill=(*GREEN, 225))

    yy += 32

    d.text((x0 + 22, yy), "Remnant:", font=font, fill=(*CYAN2, 170))
    d.text((x0 + 98, yy), remnant, font=font, fill=(*GREEN, 225))



VISUAL_TOTAL_FRAMES = sum(segment["frames"] for segment in VISUAL_TIMELINE)

def is_visual_hold_frame(frame_index: int) -> bool:
    frame_index = frame_index % TOTAL_FRAMES
    cursor = 0

    for segment in VISUAL_TIMELINE:
        start = cursor
        end = cursor + segment["frames"]

        if start <= frame_index < end:
            return segment["type"] == "hold"

        cursor = end

    return False

def draw_hr_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)

    font_big = load_font(20)
    font_mid = load_font(19)
    font_small = load_font(15)
    font_tick = load_font(15)
    font_axis = load_font(17)

    phase = 2 * np.pi * i / TOTAL_FRAMES
    progress = visual_progress_with_stage_holds(i)

    age, stage_label, log_t, log_l, radius, color, local_stage = track_state(progress)

    draw_hud_frame(frame, font_big)
    draw_grid_background(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))

    draw_hr_axes(scene, font_axis, font_tick)
    draw_star_cloud(scene)
    draw_region_labels(scene, font_mid)

    draw_evolution_trail(scene, progress)

    x, y = hr_map(log_t, log_l)
    draw_track_marker(scene, x, y, phase)


    if is_visual_hold_frame(i):
        d = ImageDraw.Draw(scene)

        label_lines = wrap_text(stage_label, 18).split("\n")

        box_x0 = x + 32
        box_y0 = y - 22
        box_x1 = box_x0 + 185
        box_y1 = box_y0 + 24 + 18 * len(label_lines)

        # чтобы подпись ЧД не вылезала за экран
        if box_x1 > 840:
            box_x0 = x - 220
            box_x1 = box_x0 + 185

        if box_y0 < 90:
            box_y0 = y + 24
            box_y1 = box_y0 + 24 + 18 * len(label_lines)

        d.rectangle(
            [box_x0, box_y0, box_x1, box_y1],
            fill=(*BG_DARK, 210),
            outline=(*GREEN, 180),
            width=1,
        )

        yy = box_y0 + 10

        for line in label_lines:
            d.text(
                (box_x0 + 10, yy),
                line,
                font=font_small,
                fill=(*GREEN, 230),
            )
            yy += 18


    draw_stage_timeline(scene, progress, font_small)

    temp = 10 ** log_t
    lum = 10 ** log_l

    metrics_rows = [
        ("mass", "20 Msun"),
        ("age", f"{age:.2f} Myr"),
        ("T_eff", f"{temp:,.0f} K"),
        ("log L", f"{log_l:+.2f}"),
        ("radius", f"{max(1, radius * 12):.0f} Rsun"),
        ("fate", "black hole"),
    ]

    draw_empty_star_preview_panel(
        scene,
        (900, 53, 1155, 256),
        font_small,
        radius,
        color,
        phase,
        stage_label,
        local_stage,
    )

    draw_star_evolution_panel(scene, (900, 265, 1155, 435), stage_label, progress, font_small)
    draw_star_metrics_panel(scene, (900, 450, 1155, 670), metrics_rows, font_small)

    if stage_label == "BLACK HOLE REMNANT":
        overlay = Image.new("RGBA", frame.size, (0, 0, 0, 0))
        od = ImageDraw.Draw(overlay)
        alpha = int(120 * local_stage)
        od.rectangle([82, 128, 835, 525], fill=(0, 0, 0, alpha))
        scene.alpha_composite(overlay)

    glow_composite(frame, scene, blur=4)
    return frame


def draw_star_symbol(
    layer: Image.Image,
    x: float,
    y: float,
    radius: float,
    color,
    phase: float,
    label: str,
    local_stage: float,
    scale: float = 1.0,
):
    d = ImageDraw.Draw(layer)

    radius *= scale
    pulse = 0.82 + 0.18 * np.sin(phase * 5) ** 2

    if label == "BLACK HOLE REMNANT":
        bh_r = radius * (1.0 + 1.2 * local_stage)
        d.ellipse(
            [x - bh_r * 2.2, y - bh_r * 2.2, x + bh_r * 2.2, y + bh_r * 2.2],
            fill=(0, 0, 0, int(90 + 140 * local_stage)),
        )
        d.ellipse(
            [x - bh_r, y - bh_r, x + bh_r, y + bh_r],
            fill=(0, 0, 0, 255),
            outline=(*PURPLE, 190),
            width=2,
        )
        d.arc(
            [x - bh_r * 1.6, y - bh_r * 0.7, x + bh_r * 1.6, y + bh_r * 0.7],
            0,
            360,
            fill=(*ORANGE, 200),
            width=3,
        )
        return

    if label == "RED SUPERGIANT":
        shell_r = radius * (1.35 + 0.08 * np.sin(phase * 6))
        d.ellipse(
            [x - shell_r, y - shell_r, x + shell_r, y + shell_r],
            fill=(*RED, 38),
            outline=(*ORANGE, 90),
            width=2,
        )

    if label == "CORE COLLAPSE":
        shock_r = radius * (1.2 + 1.8 * local_stage)
        d.ellipse(
            [x - shock_r, y - shock_r, x + shock_r, y + shock_r],
            outline=(*WHITE, int(230 * (1 - local_stage))),
            width=4,
        )

    for k, (s, alpha) in enumerate([(2.6, 22), (1.8, 48), (1.25, 85)]):
        rr = radius * s * pulse
        d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(*color, alpha))

    d.ellipse(
        [x - radius, y - radius, x + radius, y + radius],
        fill=(*color, 235),
        outline=(*WHITE, 220),
        width=2,
    )

def draw_track_marker(layer: Image.Image, x: float, y: float, phase: float):
    d = ImageDraw.Draw(layer)

    pulse = 0.6 + 0.4 * np.sin(phase * 5) ** 2
    r = 8
    ring = 24 + 4 * pulse

    d.ellipse(
        [x - ring, y - ring, x + ring, y + ring],
        outline=(*GREEN, int(150 + 80 * pulse)),
        width=2,
    )
    d.line([x - 34, y, x - 12, y], fill=(*GREEN, 190), width=1)
    d.line([x + 12, y, x + 34, y], fill=(*GREEN, 190), width=1)
    d.line([x, y - 34, x, y - 12], fill=(*GREEN, 190), width=1)
    d.line([x, y + 12, x, y + 34], fill=(*GREEN, 190), width=1)

    d.ellipse(
        [x - r, y - r, x + r, y + r],
        fill=(*WHITE, 230),
        outline=(*GREEN, 230),
        width=1,
    )

def draw_empty_star_preview_panel(
    layer,
    box,
    font,
    radius,
    color,
    phase,
    stage_label,
    local_stage,
):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 18), "STAR VIEW", font=font, fill=(*CYAN2, 190))

    cx = (x0 + x1) / 2
    cy = y0 + 108

    panel_w = x1 - x0
    panel_h = y1 - y0

    max_preview_radius = min(panel_w * 0.18, panel_h * 0.16)

    if "BLACK HOLE" in stage_label:
        scaled_radius = max_preview_radius * 0.42
    elif "RED SUPERGIANT" in stage_label:
        scaled_radius = max_preview_radius * 0.95
    elif "CORE COLLAPSE" in stage_label:
        scaled_radius = max_preview_radius * 0.82
    else:
        scaled_radius = min(radius * 0.78, max_preview_radius)

    draw_star_symbol(
        layer=layer,
        x=cx,
        y=cy,
        radius=scaled_radius,
        color=color,
        phase=phase,
        label=stage_label,
        local_stage=local_stage,
        scale=1.0,
    )


def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "34",
            "-pix_fmt", "yuva420p",
            "-auto-alt-ref", "0",
            "-row-mt", "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "22",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str) -> Path:
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] stellar evolution track: very massive star")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")
        frames.append(draw_hr_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(f"Created 1 animation(s) as '{output_format}' in '{ANIMATIONS_DIR.resolve()}'")


main()

[START] stellar evolution track: very massive star
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 570
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/570
[GENERATE] frame 24/570
[GENERATE] frame 48/570
[GENERATE] frame 72/570
[GENERATE] frame 96/570
[GENERATE] frame 120/570
[GENERATE] frame 144/570
[GENERATE] frame 168/570
[GENERATE] frame 192/570
[GENERATE] frame 216/570
[GENERATE] frame 240/570
[GENERATE] frame 264/570
[GENERATE] frame 288/570
[GENERATE] frame 312/570
[GENERATE] frame 336/570
[GENERATE] frame 360/570
[GENERATE] frame 384/570
[GENERATE] frame 408/570
[GENERATE] frame 432/570
[GENERATE] frame 456/570
[GENERATE] frame 480/570
[GENERATE] frame 504/570
[GENERATE] frame 528/570
[GENERATE] frame 552/570


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] stellar_evolution_solar_star
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/stellar_evolution_solar_star/stellar_evolution_solar_star.webm

Created 1 animation(s) as 'webm' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


# Sun

In [37]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import textwrap
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


OUTPUT_FORMAT = "webm"  # "webm" | "gif" | "mp4"
FPS = 24

TRACK = [
    {
        "label": "PROTOSTAR",
        "t0": 0.0,
        "t1": 0.03,
        "logT0": 3.52,
        "logL0": 1.9,
        "logT1": 3.60,
        "logL1": 1.2,
        "r0": 36,
        "r1": 28,
        "c0": (255, 120, 70),
        "c1": (255, 170, 90),
    },

    {
        "label": "T-TAURI STAR",
        "t0": 0.03,
        "t1": 0.08,
        "logT0": 3.60,
        "logL0": 1.2,
        "logT1": 3.76,
        "logL1": 0.00,
        "r0": 28,
        "r1": 10,
        "c0": (255, 170, 90),
        "c1": (255, 240, 120),
    },
    {
        "label": "SOLAR MAIN SEQUENCE",
        "t0": 0.08,
        "t1": 10.0,
        "logT0": 3.76,
        "logL0": 0.00,
        "logT1": 3.74,
        "logL1": 0.18,
        "r0": 10,
        "r1": 12,
        "c0": (255, 240, 120),
        "c1": (255, 220, 90),
    },


    {
        "label": "RED GIANT",
        "t0": 10.0,
        "t1": 11.2,
        "logT0": 3.74,
        "logL0": 0.18,
        "logT1": 3.56,
        "logL1": 3.0,
        "r0": 12,
        "r1": 48,
        "c0": (255, 220, 90),
        "c1": (255, 90, 60),
    },

    {
        "label": "ASYMPTOTIC GIANT BRANCH",
        "t0": 11.2,
        "t1": 11.8,
        "logT0": 3.56,
        "logL0": 3.0,
        "logT1": 3.52,
        "logL1": 3.6,
        "r0": 48,
        "r1": 62,
        "c0": (255, 90, 60),
        "c1": (255, 60, 40),
    },

    {
        "label": "PLANETARY NEBULA",
        "t0": 11.8,
        "t1": 12.05,
        "logT0": 3.52,
        "logL0": 3.6,
        "logT1": 4.58,
        "logL1": 2.2,
        "r0": 62,
        "r1": 20,
        "c0": (255, 230, 180),
        "c1": (120, 220, 255),
    },
    {
        "label": "POST-AGB CORE",
        "t0": 12.05,
        "t1": 12.25,
        "logT0": 4.58,
        "logL0": 2.2,
        "logT1": 4.58,
        "logL1": 0.8,
        "r0": 20,
        "r1": 9,
        "c0": (120, 220, 255),
        "c1": (210, 240, 255),
    },
    {
        "label": "WHITE DWARF",
        "t0": 12.25,
        "t1": 13.0,
        "logT0": 4.58,
        "logL0": 0.8,
        "logT1": 4.08,
        "logL1": -2.8,
        "r0": 9,
        "r1": 6,
        "c0": (210, 240, 255),
        "c1": (180, 210, 255),
    },

]

ANIMATIONS_DIR = Path("media-site/animations")
ANIMATION_NAME = "stellar_evolution_solar_like_star"

CANVAS_SIZE = (1200, 720)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)
BLACK = (0, 0, 0)

TOTAL_MYR = 13.0
MOVE_FRAMES = 42
HOLD_FRAMES = FPS * 3


def build_visual_timeline(
    move_frames: int = MOVE_FRAMES,
    hold_frames: int = HOLD_FRAMES,
):
    segments = []

    for stage in TRACK:
        p0 = stage["t0"] / TOTAL_MYR
        p1 = stage["t1"] / TOTAL_MYR

        segments.append({
            "type": "move",
            "p0": p0,
            "p1": p1,
            "frames": move_frames,
        })

        segments.append({
            "type": "hold",
            "p0": p1,
            "p1": p1,
            "frames": hold_frames,
        })

    return segments


VISUAL_TIMELINE = build_visual_timeline()
TOTAL_FRAMES = sum(segment["frames"] for segment in VISUAL_TIMELINE)


def visual_progress_with_stage_holds(frame_index: int) -> float:
    frame_index = frame_index % TOTAL_FRAMES
    cursor = 0

    for segment in VISUAL_TIMELINE:
        start = cursor
        end = cursor + segment["frames"]

        if start <= frame_index < end:
            if segment["type"] == "hold":
                return segment["p0"]

            local = (frame_index - start) / max(1, segment["frames"] - 1)
            local = smoothstep(local)

            return lerp(segment["p0"], segment["p1"], local)

        cursor = end

    return 1.0


def is_visual_hold_frame(frame_index: int) -> bool:
    frame_index = frame_index % TOTAL_FRAMES
    cursor = 0

    for segment in VISUAL_TIMELINE:
        start = cursor
        end = cursor + segment["frames"]

        if start <= frame_index < end:
            return segment["type"] == "hold"

        cursor = end

    return False


def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 4):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t: float) -> float:
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3 - 2 * t)


def lerp(a, b, t):
    return a + (b - a) * t


def mix_color(c1, c2, t):
    t = float(np.clip(t, 0, 1))
    return tuple(int(lerp(a, b, t)) for a, b in zip(c1, c2))


def wrap_lines(text: str, width: int = 16) -> list[str]:
    return textwrap.wrap(text, width=width) or [""]


def draw_cut_panel(
    d: ImageDraw.ImageDraw,
    box: tuple[int, int, int, int],
    fill=(*BG_DARK, 165),
    outline=(*CYAN, 105),
    cut: int = 18,
):
    x0, y0, x1, y1 = box

    pts = [
        (x0 + cut, y0),
        (x1, y0),
        (x1, y1 - cut),
        (x1 - cut, y1),
        (x0, y1),
        (x0, y0 + cut),
    ]

    d.polygon(pts, fill=fill)
    d.line(pts + [pts[0]], fill=outline, width=1)


def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)
    W, H = img.size

    pad = 28
    cut = 48

    pts = [
        (pad + cut, pad),
        (W - pad - 260, pad),
        (W - pad - 230, pad + 18),
        (W - pad, pad + 18),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)
    d.rectangle(
        [pad + 18, pad + 38, W - pad - 18, H - pad - 44],
        outline=(*CYAN2, 38),
        width=1,
    )

    d.text(
        (pad + 24, pad + 8),
        "TELEMETRY // STELLAR EVOLUTION TRACK // SUN-Like Star (1 Msun)",
        font=font,
        fill=(*CYAN2, 225),
    )


def draw_grid_background(img: Image.Image):
    d = ImageDraw.Draw(img)
    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 78, x, H - 64], fill=(*CYAN, 10), width=1)

    for y in range(100, H - 70, 60):
        d.line([70, y, W - 70, y], fill=(*CYAN, 9), width=1)


def hr_box():
    return 82, 128, 835, 525


def hr_map(log_temp: float, log_lum: float) -> tuple[float, float]:
    x0, y0, x1, y1 = hr_box()

    t_min, t_max = 3.35, 4.65
    l_min, l_max = -4.8, 6.6

    x = x0 + (t_max - log_temp) / (t_max - t_min) * (x1 - x0)
    y = y1 - (log_lum - l_min) / (l_max - l_min) * (y1 - y0)

    return x, y


def spectral_color(log_temp: float) -> tuple[int, int, int]:
    if log_temp > 4.45:
        return (110, 170, 255)
    if log_temp > 4.20:
        return (80, 220, 255)
    if log_temp > 3.98:
        return (180, 255, 255)
    if log_temp > 3.82:
        return (130, 255, 140)
    if log_temp > 3.68:
        return (255, 240, 60)
    if log_temp > 3.52:
        return (255, 150, 45)
    return (255, 60, 45)


def generate_hr_catalog(seed: int = 31415):
    rng = np.random.default_rng(seed)
    stars = []

    for _ in range(1450):
        u = rng.beta(1.25, 1.15)
        log_t = lerp(4.52, 3.42, u) + rng.normal(0, 0.045)
        log_l = lerp(5.3, -3.2, u) + rng.normal(0, 0.42)
        r = rng.uniform(0.7, 1.7)
        stars.append(("main", log_t, log_l, r))

    for _ in range(420):
        u = rng.random()
        log_t = lerp(3.72, 3.45, u) + rng.normal(0, 0.035)
        log_l = lerp(1.0, 4.1, u) + rng.normal(0, 0.35)
        r = rng.uniform(1.0, 2.2)
        stars.append(("giant", log_t, log_l, r))

    for _ in range(220):
        log_t = rng.uniform(3.55, 4.15)
        log_l = rng.normal(5.7, 0.28)
        r = rng.uniform(1.2, 2.6)
        stars.append(("supergiant", log_t, log_l, r))

    for _ in range(360):
        u = rng.random()
        log_t = lerp(4.42, 3.78, u) + rng.normal(0, 0.04)
        log_l = lerp(-1.0, -3.8, u) + rng.normal(0, 0.28)
        r = rng.uniform(0.6, 1.4)
        stars.append(("wd", log_t, log_l, r))

    rng.shuffle(stars)
    return stars


STAR_CATALOG = generate_hr_catalog()


def draw_hr_axes(layer: Image.Image, font_axis, font_tick):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = hr_box()

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 120))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 110), width=1)

    temp_ticks = [
        (4.60, "40000 K"),
        (4.30, "20000 K"),
        (4.00, "10000 K"),
        (3.70, "5000 K"),
        (3.40, "2500 K"),
    ]

    for log_t, label in temp_ticks:
        x, _ = hr_map(log_t, -4.8)
        d.line([x, y0, x, y1], fill=(*CYAN, 26), width=1)
        d.text((x - 36, y1 + 12), label, font=font_tick, fill=(*CYAN2, 185))

    lum_ticks = [
        (6, "10^6"),
        (4, "10^4"),
        (2, "10^2"),
        (0, "1"),
        (-2, "10^-2"),
        (-4, "10^-4"),
    ]

    for log_l, label in lum_ticks:
        _, y = hr_map(3.35, log_l)
        d.line([x0, y, x1, y], fill=(*CYAN, 24), width=1)
        d.text((x0 - 72, y - 9), label, font=font_tick, fill=(*CYAN2, 185))

    d.text((x0 + 8, y0 - 30), "Luminosity  L / Lsun", font=font_axis, fill=(*CYAN2, 215))
    d.text((x0 + 285, y1 + 44), "Temperature  T_eff, K", font=font_axis, fill=(*CYAN2, 215))


def draw_region_labels(layer: Image.Image, font_mid):
    d = ImageDraw.Draw(layer)

    d.text((236, 314), "Main Sequence", font=font_mid, fill=(*CYAN2, 150))
    d.text((650, 142), "Supergiants", font=font_mid, fill=(*CYAN2, 165))
    d.text((672, 292), "Giants", font=font_mid, fill=(*CYAN2, 150))
    d.text((286, 445), "White Dwarfs", font=font_mid, fill=(*CYAN2, 145))


def draw_star_cloud(layer: Image.Image):
    d = ImageDraw.Draw(layer)

    for idx, (_kind, log_t, log_l, radius) in enumerate(STAR_CATALOG):
        x, y = hr_map(log_t, log_l)
        color = spectral_color(log_t)
        alpha = 38 if idx % 7 else 72

        d.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            fill=(*color, alpha),
        )


def track_state(progress: float):
    age_gyr = progress * TOTAL_MYR

    for stage in TRACK:
        if stage["t0"] <= age_gyr <= stage["t1"]:
            local = (age_gyr - stage["t0"]) / max(1e-9, stage["t1"] - stage["t0"])
            local_s = smoothstep(local)

            log_t = lerp(stage["logT0"], stage["logT1"], local_s)
            log_l = lerp(stage["logL0"], stage["logL1"], local_s)
            radius = lerp(stage["r0"], stage["r1"], local_s)
            color = mix_color(stage["c0"], stage["c1"], local_s)

            return age_gyr, stage["label"], log_t, log_l, radius, color, local

    stage = TRACK[-1]
    return TOTAL_MYR, stage["label"], stage["logT1"], stage["logL1"], stage["r1"], stage["c1"], 1.0


def sampled_track_points(progress: float, n: int = 180):
    pts = []
    max_age_gyr = progress * TOTAL_MYR

    for age_gyr in np.linspace(0, max_age_gyr, n):
        p = age_gyr / TOTAL_MYR
        _, _label, log_t, log_l, radius, color, _local = track_state(p)
        x, y = hr_map(log_t, log_l)
        pts.append((x, y, radius, color))

    return pts


def draw_evolution_trail(layer: Image.Image, progress: float):
    trail = Image.new("RGBA", layer.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(trail)

    pts = sampled_track_points(progress, n=220)

    if len(pts) < 2:
        return

    for idx in range(1, len(pts)):
        x0, y0, _r0, c0 = pts[idx - 1]
        x1, y1, _r1, c1 = pts[idx]

        alpha = int(40 + 185 * idx / len(pts))
        width = int(5 + 7 * idx / len(pts))
        d.line([x0, y0, x1, y1], fill=(*c1, alpha), width=width)

    glow = trail.filter(ImageFilter.GaussianBlur(6))
    layer.alpha_composite(glow)
    layer.alpha_composite(trail)


def draw_stage_timeline(layer: Image.Image, progress: float, font):
    d = ImageDraw.Draw(layer)

    x0, y0, x1, y1 = 110, 614, 835, 644
    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 155), outline=(*CYAN, 90), width=1)

    for stage in TRACK:
        xa = x0 + (stage["t0"] / TOTAL_MYR) * (x1 - x0)
        xb = x0 + (stage["t1"] / TOTAL_MYR) * (x1 - x0)
        col = stage["c1"]
        d.rectangle([xa, y0, xb, y1], fill=(*col, 120))
        d.line([xa, y0, xa, y1], fill=(*CYAN2, 75), width=1)

    px = x0 + progress * (x1 - x0)
    d.line([px, y0 - 8, px, y1 + 8], fill=(*WHITE, 230), width=2)

    d.text(
        (x0, y1 + 10),
        "Evolution timeline, Gyr",
        font=font,
        fill=(*CYAN2, 185),
    )


def draw_star_metrics_panel(
    layer: Image.Image,
    box: tuple[int, int, int, int],
    rows: list[tuple[str, str]],
    font,
):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 18), "STAR METRICS", font=font, fill=(*CYAN2, 190))

    y = y0 + 56

    for key, value in rows:
        d.text((x0 + 22, y), key, font=font, fill=(*CYAN2, 170))
        d.text((x0 + 122, y), value, font=font, fill=(*GREEN, 225))
        y += 28


def wrap_text(text: str, width: int) -> str:
    words = text.split()
    lines = []
    current = ""

    for word in words:
        test = word if not current else current + " " + word

        if len(test) <= width:
            current = test
        else:
            if current:
                lines.append(current)
            current = word

    if current:
        lines.append(current)

    return "\n".join(lines)


def draw_star_evolution_panel(layer, box, stage_label, progress, font):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 16), "STAR EVOLUTION", font=font, fill=(*CYAN2, 190))

    wrapped_stage = wrap_text(stage_label, 16)

    if "WHITE DWARF" in stage_label:
        remnant = "White Dwarf"
        source = "Exposed core"
    elif "PLANETARY" in stage_label:
        remnant = "White Dwarf"
        source = "Envelope ejection"
    elif "ASYMPTOTIC" in stage_label:
        remnant = "Planetary Nebula"
        source = "Shell burning"
    elif "RED GIANT" in stage_label:
        remnant = "AGB"
        source = "H shell burning"
    else:
        remnant = "Red Giant"
        source = "Hydrogen fusion"

    yy = y0 + 52

    d.text((x0 + 22, yy), "Stage:", font=font, fill=(*CYAN2, 170))
    d.multiline_text(
        (x0 + 98, yy),
        wrapped_stage,
        font=font,
        fill=(*GREEN, 225),
        spacing=3,
    )

    yy += 52

    d.text((x0 + 22, yy), "Source:", font=font, fill=(*CYAN2, 170))
    d.text((x0 + 98, yy), source, font=font, fill=(*GREEN, 225))

    yy += 32

    d.text((x0 + 22, yy), "Remnant:", font=font, fill=(*CYAN2, 170))
    d.text((x0 + 98, yy), remnant, font=font, fill=(*GREEN, 225))


def draw_track_marker(layer: Image.Image, x: float, y: float, phase: float):
    d = ImageDraw.Draw(layer)

    pulse = 0.6 + 0.4 * np.sin(phase * 5) ** 2
    r = 8
    ring = 24 + 4 * pulse

    d.ellipse(
        [x - ring, y - ring, x + ring, y + ring],
        outline=(*GREEN, int(150 + 80 * pulse)),
        width=2,
    )
    d.line([x - 34, y, x - 12, y], fill=(*GREEN, 190), width=1)
    d.line([x + 12, y, x + 34, y], fill=(*GREEN, 190), width=1)
    d.line([x, y - 34, x, y - 12], fill=(*GREEN, 190), width=1)
    d.line([x, y + 12, x, y + 34], fill=(*GREEN, 190), width=1)

    d.ellipse(
        [x - r, y - r, x + r, y + r],
        fill=(*WHITE, 230),
        outline=(*GREEN, 230),
        width=1,
    )

def draw_star_symbol(
    layer: Image.Image,
    x: float,
    y: float,
    radius: float,
    color,
    phase: float,
    label: str,
    local_stage: float,
):
    d = ImageDraw.Draw(layer)

    pulse = 0.82 + 0.18 * np.sin(phase * 5) ** 2

    # ---------------------------------------------------------
    # SUPERNOVA
    # ---------------------------------------------------------

    if "SUPERNOVA" in label:

        shock_r = radius * (0.55 + local_stage * 1.8)

        for scale, alpha in [
            (1.0, 180),
            (1.25, 110),
            (1.6, 55),
            (2.1, 24),
        ]:
            rr = shock_r * scale

            d.ellipse(
                [x - rr, y - rr, x + rr, y + rr],
                outline=(255, 220, 120, alpha),
                width=max(1, int(4 - scale)),
            )

        core_r = radius * (0.28 + 0.15 * np.sin(phase * 22) ** 2)

        d.ellipse(
            [x - core_r, y - core_r, x + core_r, y + core_r],
            fill=(255, 255, 255, 240),
        )

        return

    # ---------------------------------------------------------
    # NEUTRON STAR
    # ---------------------------------------------------------

    if "NEUTRON STAR" in label:

        for scale, alpha in [
            (3.0, 18),
            (1.8, 40),
        ]:
            rr = radius * scale

            d.ellipse(
                [x - rr, y - rr, x + rr, y + rr],
                fill=(*CYAN2, alpha),
            )

        beam_len = radius * 7

        angle = phase * 5

        dx = np.cos(angle) * beam_len
        dy = np.sin(angle) * beam_len

        d.line(
            [x - dx, y - dy, x + dx, y + dy],
            fill=(*CYAN2, 170),
            width=2,
        )

        d.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            fill=(220, 245, 255, 240),
            outline=(255, 255, 255, 220),
            width=1,
        )

        return

    # ---------------------------------------------------------
    # RED SUPERGIANT
    # ---------------------------------------------------------

    if "RED SUPERGIANT" in label:

        shell_r = radius * (1.35 + 0.05 * np.sin(phase * 4))

        d.ellipse(
            [x - shell_r, y - shell_r, x + shell_r, y + shell_r],
            fill=(*RED, 28),
            outline=(*ORANGE, 80),
            width=2,
        )

    # ---------------------------------------------------------
    # PROTOSTAR
    # ---------------------------------------------------------

    if "PROTOSTAR" in label:

        rng = np.random.default_rng(42)

        cloud_r = radius * 2.4

        for _ in range(32):

            a = rng.uniform(0, 2 * np.pi)
            rr = rng.uniform(0.1, 1.0) * cloud_r

            px = x + np.cos(a) * rr * 0.9
            py = y + np.sin(a) * rr * 0.6

            pr = rng.uniform(radius * 0.25, radius * 0.8)

            d.ellipse(
                [px - pr, py - pr, px + pr, py + pr],
                fill=(255, 120, 60, int(rng.uniform(16, 48))),
            )

    # ---------------------------------------------------------
    # STANDARD STAR
    # ---------------------------------------------------------

    for scale, alpha in [
        (2.6, 22),
        (1.8, 48),
        (1.25, 85),
    ]:
        rr = radius * scale * pulse

        d.ellipse(
            [x - rr, y - rr, x + rr, y + rr],
            fill=(*color, alpha),
        )

    d.ellipse(
        [x - radius, y - radius, x + radius, y + radius],
        fill=(*color, 235),
        outline=(*WHITE, 220),
        width=2,
    )


def draw_empty_star_preview_panel(
    layer,
    box,
    font,
    radius,
    color,
    phase,
    stage_label,
    local_stage,
):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 18), "STAR VIEW", font=font, fill=(*CYAN2, 190))

    cx = (x0 + x1) / 2
    cy = y0 + 108

    panel_w = x1 - x0
    panel_h = y1 - y0

    max_preview_radius = min(panel_w * 0.18, panel_h * 0.16)

    if "WHITE DWARF" in stage_label:
        scaled_radius = max_preview_radius * 0.28
    elif "PLANETARY" in stage_label:
        scaled_radius = max_preview_radius * 0.46
    elif "ASYMPTOTIC" in stage_label:
        scaled_radius = max_preview_radius * 0.95
    elif "RED GIANT" in stage_label:
        scaled_radius = max_preview_radius * 0.85
    else:
        scaled_radius = min(radius * 0.78, max_preview_radius)

    draw_star_symbol(
        layer=layer,
        x=cx,
        y=cy,
        radius=scaled_radius,
        color=color,
        phase=phase,
        label=stage_label,
        local_stage=local_stage,
    )


def draw_stage_popup(layer, x, y, stage_label, font):
    d = ImageDraw.Draw(layer)

    label_lines = wrap_text(stage_label, 18).split("\n")

    box_x0 = x + 32
    box_y0 = y - 22
    box_x1 = box_x0 + 185
    box_y1 = box_y0 + 24 + 18 * len(label_lines)

    if box_x1 > 840:
        box_x0 = x - 220
        box_x1 = box_x0 + 185

    if box_y0 < 90:
        box_y0 = y + 24
        box_y1 = box_y0 + 24 + 18 * len(label_lines)

    d.rectangle(
        [box_x0, box_y0, box_x1, box_y1],
        fill=(*BG_DARK, 210),
        outline=(*GREEN, 180),
        width=1,
    )

    yy = box_y0 + 10

    for line in label_lines:
        d.text(
            (box_x0 + 10, yy),
            line,
            font=font,
            fill=(*GREEN, 230),
        )
        yy += 18


def draw_hr_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)

    font_big = load_font(20)
    font_mid = load_font(19)
    font_small = load_font(15)
    font_tick = load_font(15)
    font_axis = load_font(17)

    phase = 2 * np.pi * i / TOTAL_FRAMES
    progress = visual_progress_with_stage_holds(i)

    age_gyr, stage_label, log_t, log_l, radius, color, local_stage = track_state(progress)

    draw_hud_frame(frame, font_big)
    draw_grid_background(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))

    draw_hr_axes(scene, font_axis, font_tick)
    draw_star_cloud(scene)
    draw_region_labels(scene, font_mid)

    draw_evolution_trail(scene, progress)

    x, y = hr_map(log_t, log_l)
    draw_track_marker(scene, x, y, phase)

    if is_visual_hold_frame(i):
        draw_stage_popup(scene, x, y, stage_label, font_small)

    draw_stage_timeline(scene, progress, font_small)

    temp = 10 ** log_t
    lum = 10 ** log_l

    metrics_rows = [
        ("mass", "1.0 Msun"),
        ("age", f"{age_gyr:.2f} Gyr"),
        ("T_eff", f"{temp:,.0f} K"),
        ("log L", f"{log_l:+.2f}"),
        ("radius", f"{max(0.01, radius * 0.12):.2f} Rsun"),
        ("fate", "white dwarf"),
    ]

    draw_empty_star_preview_panel(
        scene,
        (900, 53, 1155, 256),
        font_small,
        radius,
        color,
        phase,
        stage_label,
        local_stage,
    )

    draw_star_evolution_panel(scene, (900, 265, 1155, 435), stage_label, progress, font_small)
    draw_star_metrics_panel(scene, (900, 450, 1155, 670), metrics_rows, font_small)

    glow_composite(frame, scene, blur=4)
    return frame


def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "34",
            "-pix_fmt", "yuva420p",
            "-auto-alt-ref", "0",
            "-row-mt", "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "22",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str) -> Path:
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] stellar evolution track: solar-like star")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")
        frames.append(draw_hr_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(f"Created 1 animation(s) as '{output_format}' in '{ANIMATIONS_DIR.resolve()}'")


main()

[START] stellar evolution track: solar-like star
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 912
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/912
[GENERATE] frame 24/912
[GENERATE] frame 48/912
[GENERATE] frame 72/912
[GENERATE] frame 96/912
[GENERATE] frame 120/912
[GENERATE] frame 144/912
[GENERATE] frame 168/912
[GENERATE] frame 192/912
[GENERATE] frame 216/912
[GENERATE] frame 240/912
[GENERATE] frame 264/912
[GENERATE] frame 288/912
[GENERATE] frame 312/912
[GENERATE] frame 336/912
[GENERATE] frame 360/912
[GENERATE] frame 384/912
[GENERATE] frame 408/912
[GENERATE] frame 432/912
[GENERATE] frame 456/912
[GENERATE] frame 480/912
[GENERATE] frame 504/912
[GENERATE] frame 528/912
[GENERATE] frame 552/912
[GENERATE] frame 576/912
[GENERATE] frame 600/912
[GENERATE] frame 624/912
[GENERATE] frame 648/912
[GENERATE] frame 672/912
[GENERATE] frame 696/912
[GENERATE] fra

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] stellar_evolution_solar_like_star
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/stellar_evolution_solar_like_star/stellar_evolution_solar_like_star.webm

Created 1 animation(s) as 'webm' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


# Heavy Star Evolution

In [39]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import textwrap
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


OUTPUT_FORMAT = "webm"
FPS = 24

TOTAL_MYR = 0.028

TRACK = [
    {
        "label": "PROTOSTAR",
        "t0": 0.0,
        "t1": 0.0015,
        "logT0": 3.62,
        "logL0": 3.2,
        "logT1": 4.18,
        "logL1": 4.6,
        "r0": 42,
        "r1": 24,
        "c0": (255, 140, 90),
        "c1": (170, 220, 255),
    },

    {
        "label": "O/B MAIN SEQUENCE",
        "t0": 0.0015,
        "t1": 0.020,
        "logT0": 4.18,
        "logL0": 4.6,
        "logT1": 4.32,
        "logL1": 5.05,
        "r0": 24,
        "r1": 28,
        "c0": (170, 220, 255),
        "c1": (230, 240, 255),
    },

    {
        "label": "BLUE SUPERGIANT",
        "t0": 0.020,
        "t1": 0.023,
        "logT0": 4.32,
        "logL0": 5.05,
        "logT1": 4.08,
        "logL1": 5.30,
        "r0": 28,
        "r1": 38,
        "c0": (230, 240, 255),
        "c1": (180, 230, 255),
    },

    {
        "label": "RED SUPERGIANT",
        "t0": 0.023,
        "t1": 0.026,
        "logT0": 4.08,
        "logL0": 5.30,
        "logT1": 3.55,
        "logL1": 5.45,
        "r0": 38,
        "r1": 74,
        "c0": (255, 210, 120),
        "c1": (255, 70, 40),
    },

    {
        "label": "CORE COLLAPSE",
        "t0": 0.026,
        "t1": 0.0264,
        "logT0": 3.55,
        "logL0": 5.45,
        "logT1": 3.58,
        "logL1": 5.75,
        "r0": 74,
        "r1": 30,
        "c0": (255, 120, 80),
        "c1": (255, 255, 255),
    },

    {
        "label": "SUPERNOVA",
        "t0": 0.0264,
        "t1": 0.0269,
        "logT0": 3.58,
        "logL0": 5.75,
        "logT1": 3.58,
        "logL1": 6.30,
        "r0": 30,
        "r1": 120,
        "c0": (255, 255, 255),
        "c1": (255, 220, 120),
    },

    {
        "label": "NEUTRON STAR",
        "t0": 0.0269,
        "t1": 0.028,
        "logT0": 3.58,
        "logL0": 6.30,
        "logT1": 4.55,
        "logL1": -4.2,
        "r0": 18,
        "r1": 5,
        "c0": (255, 255, 255),
        "c1": (170, 220, 255),
    },
]

ANIMATIONS_DIR = Path("media-site/animations")
ANIMATION_NAME = "stellar_evolution_heavy_star"

CANVAS_SIZE = (1200, 720)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)

MOVE_FRAMES = 42
HOLD_FRAMES = FPS * 3


def smoothstep(t: float) -> float:
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3 - 2 * t)


def lerp(a, b, t):
    return a + (b - a) * t


def mix_color(c1, c2, t):
    t = float(np.clip(t, 0, 1))
    return tuple(int(lerp(a, b, t)) for a, b in zip(c1, c2))


def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 4):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def build_visual_timeline():
    segments = []

    for stage in TRACK:
        p0 = stage["t0"] / TOTAL_MYR
        p1 = stage["t1"] / TOTAL_MYR

        segments.append({
            "type": "move",
            "p0": p0,
            "p1": p1,
            "frames": MOVE_FRAMES,
        })

        segments.append({
            "type": "hold",
            "p0": p1,
            "p1": p1,
            "frames": HOLD_FRAMES,
        })

    return segments


VISUAL_TIMELINE = build_visual_timeline()
TOTAL_FRAMES = sum(s["frames"] for s in VISUAL_TIMELINE)


def visual_progress_with_stage_holds(frame_index: int) -> float:
    frame_index %= TOTAL_FRAMES
    cursor = 0

    for segment in VISUAL_TIMELINE:
        start = cursor
        end = cursor + segment["frames"]

        if start <= frame_index < end:

            if segment["type"] == "hold":
                return segment["p0"]

            local = (frame_index - start) / max(1, segment["frames"] - 1)
            local = smoothstep(local)

            return lerp(segment["p0"], segment["p1"], local)

        cursor = end

    return 1.0


def is_visual_hold_frame(frame_index: int) -> bool:
    frame_index %= TOTAL_FRAMES
    cursor = 0

    for segment in VISUAL_TIMELINE:
        start = cursor
        end = cursor + segment["frames"]

        if start <= frame_index < end:
            return segment["type"] == "hold"

        cursor = end

    return False


def draw_cut_panel(
    d: ImageDraw.ImageDraw,
    box,
    fill=(*BG_DARK, 165),
    outline=(*CYAN, 105),
    cut: int = 18,
):
    x0, y0, x1, y1 = box

    pts = [
        (x0 + cut, y0),
        (x1, y0),
        (x1, y1 - cut),
        (x1 - cut, y1),
        (x0, y1),
        (x0, y0 + cut),
    ]

    d.polygon(pts, fill=fill)
    d.line(pts + [pts[0]], fill=outline, width=1)


def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)

    W, H = img.size
    pad = 28
    cut = 48

    pts = [
        (pad + cut, pad),
        (W - pad - 260, pad),
        (W - pad - 230, pad + 18),
        (W - pad, pad + 18),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)

    d.rectangle(
        [pad + 18, pad + 38, W - pad - 18, H - pad - 44],
        outline=(*CYAN2, 38),
        width=1,
    )

    d.text(
        (pad + 24, pad + 8),
        "TELEMETRY // STELLAR EVOLUTION TRACK // HEAVY STAR (15 Msun)",
        font=font,
        fill=(*CYAN2, 225),
    )


def draw_grid_background(img: Image.Image):
    d = ImageDraw.Draw(img)

    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 78, x, H - 64], fill=(*CYAN, 10), width=1)

    for y in range(100, H - 70, 60):
        d.line([70, y, W - 70, y], fill=(*CYAN, 9), width=1)


def hr_box():
    return 82, 128, 835, 525


def hr_map(log_temp: float, log_lum: float):
    x0, y0, x1, y1 = hr_box()

    t_min, t_max = 3.35, 4.65
    l_min, l_max = -4.8, 6.6

    x = x0 + (t_max - log_temp) / (t_max - t_min) * (x1 - x0)
    y = y1 - (log_lum - l_min) / (l_max - l_min) * (y1 - y0)

    return x, y


def spectral_color(log_temp: float):
    if log_temp > 4.45:
        return (110, 170, 255)
    if log_temp > 4.20:
        return (80, 220, 255)
    if log_temp > 3.98:
        return (180, 255, 255)
    if log_temp > 3.82:
        return (130, 255, 140)
    if log_temp > 3.68:
        return (255, 240, 60)
    if log_temp > 3.52:
        return (255, 150, 45)
    return (255, 60, 45)

def generate_hr_catalog(seed: int = 31415):
    rng = np.random.default_rng(seed)
    stars = []

    for _ in range(1450):
        u = rng.beta(1.25, 1.15)
        log_t = lerp(4.52, 3.42, u) + rng.normal(0, 0.045)
        log_l = lerp(5.3, -3.2, u) + rng.normal(0, 0.42)
        r = rng.uniform(0.7, 1.7)
        stars.append(("main", log_t, log_l, r))

    for _ in range(420):
        u = rng.random()
        log_t = lerp(3.72, 3.45, u) + rng.normal(0, 0.035)
        log_l = lerp(1.0, 4.1, u) + rng.normal(0, 0.35)
        r = rng.uniform(1.0, 2.2)
        stars.append(("giant", log_t, log_l, r))

    for _ in range(220):
        log_t = rng.uniform(3.55, 4.15)
        log_l = rng.normal(5.7, 0.28)
        r = rng.uniform(1.2, 2.6)
        stars.append(("supergiant", log_t, log_l, r))

    for _ in range(360):
        u = rng.random()
        log_t = lerp(4.42, 3.78, u) + rng.normal(0, 0.04)
        log_l = lerp(-1.0, -3.8, u) + rng.normal(0, 0.28)
        r = rng.uniform(0.6, 1.4)
        stars.append(("wd", log_t, log_l, r))

    rng.shuffle(stars)
    return stars


STAR_CATALOG = generate_hr_catalog()


def draw_hr_axes(layer: Image.Image, font_axis, font_tick):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = hr_box()

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 120))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 110), width=1)

    temp_ticks = [
        (4.60, "40000 K"),
        (4.30, "20000 K"),
        (4.00, "10000 K"),
        (3.70, "5000 K"),
        (3.40, "2500 K"),
    ]

    for log_t, label in temp_ticks:
        x, _ = hr_map(log_t, -4.8)
        d.line([x, y0, x, y1], fill=(*CYAN, 26), width=1)
        d.text((x - 36, y1 + 12), label, font=font_tick, fill=(*CYAN2, 185))

    lum_ticks = [
        (6, "10^6"),
        (4, "10^4"),
        (2, "10^2"),
        (0, "1"),
        (-2, "10^-2"),
        (-4, "10^-4"),
    ]

    for log_l, label in lum_ticks:
        _, y = hr_map(3.35, log_l)
        d.line([x0, y, x1, y], fill=(*CYAN, 24), width=1)
        d.text((x0 - 72, y - 9), label, font=font_tick, fill=(*CYAN2, 185))

    d.text((x0 + 8, y0 - 30), "Luminosity  L / Lsun", font=font_axis, fill=(*CYAN2, 215))
    d.text((x0 + 285, y1 + 44), "Temperature  T_eff, K", font=font_axis, fill=(*CYAN2, 215))


def draw_region_labels(layer: Image.Image, font_mid):
    d = ImageDraw.Draw(layer)
    d.text((236, 314), "Main Sequence", font=font_mid, fill=(*CYAN2, 150))
    d.text((650, 142), "Supergiants", font=font_mid, fill=(*CYAN2, 165))
    d.text((672, 292), "Giants", font=font_mid, fill=(*CYAN2, 150))
    d.text((286, 445), "White Dwarfs", font=font_mid, fill=(*CYAN2, 145))


def draw_star_cloud(layer: Image.Image):
    d = ImageDraw.Draw(layer)

    for idx, (_kind, log_t, log_l, radius) in enumerate(STAR_CATALOG):
        x, y = hr_map(log_t, log_l)
        color = spectral_color(log_t)
        alpha = 38 if idx % 7 else 72

        d.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            fill=(*color, alpha),
        )


def track_state(progress: float):
    age_myr = progress * TOTAL_MYR

    for stage in TRACK:
        if stage["t0"] <= age_myr <= stage["t1"]:
            local = (age_myr - stage["t0"]) / max(1e-9, stage["t1"] - stage["t0"])
            local_s = smoothstep(local)

            log_t = lerp(stage["logT0"], stage["logT1"], local_s)
            log_l = lerp(stage["logL0"], stage["logL1"], local_s)
            radius = lerp(stage["r0"], stage["r1"], local_s)
            color = mix_color(stage["c0"], stage["c1"], local_s)

            return age_myr, stage["label"], log_t, log_l, radius, color, local

    stage = TRACK[-1]
    return TOTAL_MYR, stage["label"], stage["logT1"], stage["logL1"], stage["r1"], stage["c1"], 1.0


def sampled_track_points(progress: float, n: int = 220):
    pts = []
    max_age_myr = progress * TOTAL_MYR

    for age_myr in np.linspace(0, max_age_myr, n):
        p = age_myr / TOTAL_MYR
        _, _label, log_t, log_l, radius, color, _local = track_state(p)
        x, y = hr_map(log_t, log_l)
        pts.append((x, y, radius, color))

    return pts


def draw_evolution_trail(layer: Image.Image, progress: float):
    trail = Image.new("RGBA", layer.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(trail)

    pts = sampled_track_points(progress, n=240)

    if len(pts) < 2:
        return

    for idx in range(1, len(pts)):
        x0, y0, _r0, _c0 = pts[idx - 1]
        x1, y1, _r1, c1 = pts[idx]

        alpha = int(40 + 185 * idx / len(pts))
        width = int(5 + 7 * idx / len(pts))
        d.line([x0, y0, x1, y1], fill=(*c1, alpha), width=width)

    glow = trail.filter(ImageFilter.GaussianBlur(6))
    layer.alpha_composite(glow)
    layer.alpha_composite(trail)


def draw_stage_timeline(layer: Image.Image, progress: float, font):
    d = ImageDraw.Draw(layer)

    x0, y0, x1, y1 = 110, 614, 835, 644
    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 155), outline=(*CYAN, 90), width=1)

    for stage in TRACK:
        xa = x0 + (stage["t0"] / TOTAL_MYR) * (x1 - x0)
        xb = x0 + (stage["t1"] / TOTAL_MYR) * (x1 - x0)
        col = stage["c1"]
        d.rectangle([xa, y0, xb, y1], fill=(*col, 120))
        d.line([xa, y0, xa, y1], fill=(*CYAN2, 75), width=1)

    px = x0 + progress * (x1 - x0)
    d.line([px, y0 - 8, px, y1 + 8], fill=(*WHITE, 230), width=2)

    d.text(
        (x0, y1 + 10),
        "Evolution timeline, Myr",
        font=font,
        fill=(*CYAN2, 185),
    )


def draw_star_metrics_panel(
    layer: Image.Image,
    box,
    rows,
    font,
):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 18), "STAR METRICS", font=font, fill=(*CYAN2, 190))

    y = y0 + 56

    for key, value in rows:
        d.text((x0 + 22, y), key, font=font, fill=(*CYAN2, 170))
        d.text((x0 + 122, y), value, font=font, fill=(*GREEN, 225))
        y += 28


def draw_star_evolution_panel(layer, box, stage_label, progress, font):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 16), "STAR EVOLUTION", font=font, fill=(*CYAN2, 190))

    wrapped_stage = "\n".join(wrap_lines(stage_label, 16))

    if "NEUTRON STAR" in stage_label:
        remnant = "Neutron Star"
        source = "Collapsed core"
    elif "SUPERNOVA" in stage_label:
        remnant = "Neutron Star"
        source = "Envelope blast"
    elif "CORE COLLAPSE" in stage_label:
        remnant = "NS forming"
        source = "Iron core"
    elif "RED SUPERGIANT" in stage_label:
        remnant = "Core Collapse"
        source = "Shell burning"
    elif "BLUE SUPERGIANT" in stage_label:
        remnant = "Red Supergiant"
        source = "H/He burning"
    elif "O/B MAIN" in stage_label:
        remnant = "Supernova"
        source = "Hydrogen fusion"
    else:
        remnant = "Massive star"
        source = "Cloud collapse"

    yy = y0 + 52

    d.text((x0 + 22, yy), "Stage:", font=font, fill=(*CYAN2, 170))
    d.multiline_text(
        (x0 + 98, yy),
        wrapped_stage,
        font=font,
        fill=(*GREEN, 225),
        spacing=3,
    )

    yy += 52

    d.text((x0 + 22, yy), "Source:", font=font, fill=(*CYAN2, 170))
    d.text((x0 + 98, yy), source, font=font, fill=(*GREEN, 225))

    yy += 32

    d.text((x0 + 22, yy), "Remnant:", font=font, fill=(*CYAN2, 170))
    d.text((x0 + 98, yy), remnant, font=font, fill=(*GREEN, 225))


def draw_track_marker(layer: Image.Image, x: float, y: float, phase: float):
    d = ImageDraw.Draw(layer)

    pulse = 0.6 + 0.4 * np.sin(phase * 5) ** 2
    r = 8
    ring = 24 + 4 * pulse

    d.ellipse(
        [x - ring, y - ring, x + ring, y + ring],
        outline=(*GREEN, int(150 + 80 * pulse)),
        width=2,
    )

    d.line([x - 34, y, x - 12, y], fill=(*GREEN, 190), width=1)
    d.line([x + 12, y, x + 34, y], fill=(*GREEN, 190), width=1)
    d.line([x, y - 34, x, y - 12], fill=(*GREEN, 190), width=1)
    d.line([x, y + 12, x, y + 34], fill=(*GREEN, 190), width=1)

    d.ellipse(
        [x - r, y - r, x + r, y + r],
        fill=(*WHITE, 230),
        outline=(*GREEN, 230),
        width=1,
    )


def draw_star_symbol(
    layer: Image.Image,
    x: float,
    y: float,
    radius: float,
    color,
    phase: float,
    label: str,
    local_stage: float,
):
    d = ImageDraw.Draw(layer)
    pulse = 0.82 + 0.18 * np.sin(phase * 5) ** 2

    if "SUPERNOVA" in label:
        shock_r = radius * (0.55 + local_stage * 1.9)

        for scale, alpha, width in [
            (1.0, 220, 4),
            (1.35, 150, 3),
            (1.8, 80, 2),
            (2.35, 35, 1),
        ]:
            rr = shock_r * scale
            d.ellipse(
                [x - rr, y - rr, x + rr, y + rr],
                outline=(255, 220, 120, alpha),
                width=width,
            )

        core_r = radius * (0.18 + 0.18 * np.sin(phase * 18) ** 2)
        d.ellipse(
            [x - core_r, y - core_r, x + core_r, y + core_r],
            fill=(255, 255, 255, 245),
            outline=(255, 240, 180, 220),
            width=2,
        )
        return

    if "NEUTRON STAR" in label:
        beam_len = radius * 8
        angle = phase * 5.5

        dx = np.cos(angle) * beam_len
        dy = np.sin(angle) * beam_len

        d.line([x - dx, y - dy, x + dx, y + dy], fill=(*CYAN2, 170), width=2)
        d.line(
            [x + dy * 0.35, y - dx * 0.35, x - dy * 0.35, y + dx * 0.35],
            fill=(*PURPLE, 120),
            width=1,
        )

        for scale, alpha in [(3.0, 18), (1.8, 45)]:
            rr = radius * scale
            d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(*CYAN2, alpha))

        d.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            fill=(220, 245, 255, 245),
            outline=(255, 255, 255, 230),
            width=2,
        )
        return

    if "PROTOSTAR" in label:
        rng = np.random.default_rng(42)
        cloud_r = radius * 2.4

        for _ in range(32):
            a = rng.uniform(0, 2 * np.pi)
            rr = rng.uniform(0.1, 1.0) * cloud_r
            px = x + np.cos(a) * rr * 0.9
            py = y + np.sin(a) * rr * 0.6
            pr = rng.uniform(radius * 0.25, radius * 0.8)

            d.ellipse(
                [px - pr, py - pr, px + pr, py + pr],
                fill=(255, 120, 60, int(rng.uniform(16, 48))),
            )

    if "RED SUPERGIANT" in label:
        shell_r = radius * (1.35 + 0.06 * np.sin(phase * 4))
        d.ellipse(
            [x - shell_r, y - shell_r, x + shell_r, y + shell_r],
            fill=(*RED, 32),
            outline=(*ORANGE, 90),
            width=2,
        )

    if "CORE COLLAPSE" in label:
        shock_r = radius * (1.15 + 0.65 * local_stage)
        d.ellipse(
            [x - shock_r, y - shock_r, x + shock_r, y + shock_r],
            outline=(*WHITE, int(220 * (1.0 - local_stage))),
            width=3,
        )

    for scale, alpha in [(2.6, 22), (1.8, 48), (1.25, 85)]:
        rr = radius * scale * pulse
        d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(*color, alpha))

    d.ellipse(
        [x - radius, y - radius, x + radius, y + radius],
        fill=(*color, 235),
        outline=(*WHITE, 220),
        width=2,
    )


def draw_empty_star_preview_panel(
    layer,
    box,
    font,
    radius,
    color,
    phase,
    stage_label,
    local_stage,
):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 18), "STAR VIEW", font=font, fill=(*CYAN2, 190))

    cx = (x0 + x1) / 2
    cy = y0 + 108

    panel_w = x1 - x0
    panel_h = y1 - y0
    max_preview_radius = min(panel_w * 0.18, panel_h * 0.16)

    if "SUPERNOVA" in stage_label:
        scaled_radius = max_preview_radius * 0.72
    elif "NEUTRON STAR" in stage_label:
        scaled_radius = max_preview_radius * 0.22
    elif "CORE COLLAPSE" in stage_label:
        scaled_radius = max_preview_radius * 0.62
    elif "RED SUPERGIANT" in stage_label:
        scaled_radius = max_preview_radius * 0.95
    elif "PROTOSTAR" in stage_label:
        scaled_radius = max_preview_radius * 0.68
    else:
        scaled_radius = min(radius * 0.70, max_preview_radius)

    draw_star_symbol(
        layer=layer,
        x=cx,
        y=cy,
        radius=scaled_radius,
        color=color,
        phase=phase,
        label=stage_label,
        local_stage=local_stage,
    )


def draw_stage_popup(layer, x, y, stage_label, font):
    d = ImageDraw.Draw(layer)

    label_lines = wrap_lines(stage_label, 18)

    box_x0 = x + 32
    box_y0 = y - 22
    box_x1 = box_x0 + 185
    box_y1 = box_y0 + 24 + 18 * len(label_lines)

    if box_x1 > 840:
        box_x0 = x - 220
        box_x1 = box_x0 + 185

    if box_y0 < 90:
        box_y0 = y + 24
        box_y1 = box_y0 + 24 + 18 * len(label_lines)

    d.rectangle(
        [box_x0, box_y0, box_x1, box_y1],
        fill=(*BG_DARK, 210),
        outline=(*GREEN, 180),
        width=1,
    )

    yy = box_y0 + 10

    for line in label_lines:
        d.text((box_x0 + 10, yy), line, font=font, fill=(*GREEN, 230))
        yy += 18


def draw_hr_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)

    font_big = load_font(20)
    font_mid = load_font(19)
    font_small = load_font(15)
    font_tick = load_font(15)
    font_axis = load_font(17)

    phase = 2 * np.pi * i / TOTAL_FRAMES
    progress = visual_progress_with_stage_holds(i)

    age_myr, stage_label, log_t, log_l, radius, color, local_stage = track_state(progress)

    draw_hud_frame(frame, font_big)
    draw_grid_background(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))

    draw_hr_axes(scene, font_axis, font_tick)
    draw_star_cloud(scene)
    draw_region_labels(scene, font_mid)

    draw_evolution_trail(scene, progress)

    x, y = hr_map(log_t, log_l)
    draw_track_marker(scene, x, y, phase)

    if is_visual_hold_frame(i):
        draw_stage_popup(scene, x, y, stage_label, font_small)

    draw_stage_timeline(scene, progress, font_small)

    temp = 10 ** log_t
    lum = 10 ** log_l

    metrics_rows = [
        ("mass", "15.0 Msun"),
        ("age", f"{age_myr * 1000:.1f} kyr"),
        ("T_eff", f"{temp:,.0f} K"),
        ("log L", f"{log_l:+.2f}"),
        ("radius", f"{max(0.01, radius * 0.25):.1f} Rsun"),
        ("fate", "neutron star"),
    ]

    draw_empty_star_preview_panel(
        scene,
        (900, 53, 1155, 256),
        font_small,
        radius,
        color,
        phase,
        stage_label,
        local_stage,
    )

    draw_star_evolution_panel(scene, (900, 265, 1155, 435), stage_label, progress, font_small)
    draw_star_metrics_panel(scene, (900, 450, 1155, 670), metrics_rows, font_small)

    glow_composite(frame, scene, blur=4)
    return frame


def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False):
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "34",
            "-pix_fmt", "yuva420p",
            "-auto-alt-ref", "0",
            "-row-mt", "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False):
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "22",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str):
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] stellar evolution track: heavy star")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")
        frames.append(draw_hr_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(f"Created 1 animation(s) as '{output_format}' in '{ANIMATIONS_DIR.resolve()}'")


main()

[START] stellar evolution track: heavy star
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 798
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/798
[GENERATE] frame 24/798
[GENERATE] frame 48/798
[GENERATE] frame 72/798
[GENERATE] frame 96/798
[GENERATE] frame 120/798
[GENERATE] frame 144/798
[GENERATE] frame 168/798
[GENERATE] frame 192/798
[GENERATE] frame 216/798
[GENERATE] frame 240/798
[GENERATE] frame 264/798
[GENERATE] frame 288/798
[GENERATE] frame 312/798
[GENERATE] frame 336/798
[GENERATE] frame 360/798
[GENERATE] frame 384/798
[GENERATE] frame 408/798
[GENERATE] frame 432/798
[GENERATE] frame 456/798
[GENERATE] frame 480/798
[GENERATE] frame 504/798
[GENERATE] frame 528/798
[GENERATE] frame 552/798
[GENERATE] frame 576/798
[GENERATE] frame 600/798
[GENERATE] frame 624/798
[GENERATE] frame 648/798
[GENERATE] frame 672/798
[GENERATE] frame 696/798
[GENERATE] frame 72

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] stellar_evolution_heavy_star
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/stellar_evolution_heavy_star/stellar_evolution_heavy_star.webm

Created 1 animation(s) as 'webm' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


# Red Dwarf Evolution

In [45]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import textwrap
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


OUTPUT_FORMAT = "webm"
FPS = 24

TOTAL_MYR = 0.028

TRACK = [

    {
        "label": "PROTOSTAR",
        "t0": 0.0,
        "t1": 0.08,
        "logT0": 3.48,
        "logL0": -0.8,
        "logT1": 3.52,
        "logL1": -1.2,
        "r0": 24,
        "r1": 18,
        "c0": (255, 130, 90),
        "c1": (255, 180, 110),
    },

    {
        "label": "RED DWARF MAIN SEQUENCE",
        "t0": 0.08,
        "t1": 6000.0,
        "logT0": 3.52,
        "logL0": -1.2,
        "logT1": 3.56,
        "logL1": -0.7,
        "r0": 18,
        "r1": 20,
        "c0": (255, 180, 110),
        "c1": (255, 120, 70),
    },

    {
        "label": "BLUE DWARF",
        "t0": 6000.0,
        "t1": 7000.0,
        "logT0": 3.56,
        "logL0": -0.7,
        "logT1": 3.82,
        "logL1": -0.25,
        "r0": 20,
        "r1": 14,
        "c0": (255, 120, 70),
        "c1": (180, 220, 255),
    },

    {
        "label": "WHITE DWARF",
        "t0": 7000.0,
        "t1": 9000.0,
        "logT0": 3.82,
        "logL0": -0.25,
        "logT1": 4.25,
        "logL1": -4.0,
        "r0": 14,
        "r1": 5,
        "c0": (180, 220, 255),
        "c1": (200, 230, 255),
    },

]

TOTAL_MYR = 9000.0
ANIMATION_NAME = "stellar_evolution_red_dwarf"
ANIMATIONS_DIR = Path("media-site/animations")

CANVAS_SIZE = (1200, 720)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)

MOVE_FRAMES = 42
HOLD_FRAMES = FPS * 3


def smoothstep(t: float) -> float:
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3 - 2 * t)


def lerp(a, b, t):
    return a + (b - a) * t


def mix_color(c1, c2, t):
    t = float(np.clip(t, 0, 1))
    return tuple(int(lerp(a, b, t)) for a, b in zip(c1, c2))


def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 4):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def build_visual_timeline():
    segments = []

    for stage in TRACK:
        p0 = stage["t0"] / TOTAL_MYR
        p1 = stage["t1"] / TOTAL_MYR

        segments.append({
            "type": "move",
            "p0": p0,
            "p1": p1,
            "frames": MOVE_FRAMES,
        })

        segments.append({
            "type": "hold",
            "p0": p1,
            "p1": p1,
            "frames": HOLD_FRAMES,
        })

    return segments


VISUAL_TIMELINE = build_visual_timeline()
TOTAL_FRAMES = sum(s["frames"] for s in VISUAL_TIMELINE)


def visual_progress_with_stage_holds(frame_index: int) -> float:
    frame_index %= TOTAL_FRAMES
    cursor = 0

    for segment in VISUAL_TIMELINE:
        start = cursor
        end = cursor + segment["frames"]

        if start <= frame_index < end:

            if segment["type"] == "hold":
                return segment["p0"]

            local = (frame_index - start) / max(1, segment["frames"] - 1)
            local = smoothstep(local)

            return lerp(segment["p0"], segment["p1"], local)

        cursor = end

    return 1.0


def is_visual_hold_frame(frame_index: int) -> bool:
    frame_index %= TOTAL_FRAMES
    cursor = 0

    for segment in VISUAL_TIMELINE:
        start = cursor
        end = cursor + segment["frames"]

        if start <= frame_index < end:
            return segment["type"] == "hold"

        cursor = end

    return False


def draw_cut_panel(
    d: ImageDraw.ImageDraw,
    box,
    fill=(*BG_DARK, 165),
    outline=(*CYAN, 105),
    cut: int = 18,
):
    x0, y0, x1, y1 = box

    pts = [
        (x0 + cut, y0),
        (x1, y0),
        (x1, y1 - cut),
        (x1 - cut, y1),
        (x0, y1),
        (x0, y0 + cut),
    ]

    d.polygon(pts, fill=fill)
    d.line(pts + [pts[0]], fill=outline, width=1)


def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)

    W, H = img.size
    pad = 28
    cut = 48

    pts = [
        (pad + cut, pad),
        (W - pad - 260, pad),
        (W - pad - 230, pad + 18),
        (W - pad, pad + 18),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)

    d.rectangle(
        [pad + 18, pad + 38, W - pad - 18, H - pad - 44],
        outline=(*CYAN2, 38),
        width=1,
    )

    d.text(
        (pad + 24, pad + 8),
        "TELEMETRY // STELLAR EVOLUTION TRACK // RED DWARF (0.12 Msun)",
        font=font,
        fill=(*CYAN2, 225),
    )


def draw_grid_background(img: Image.Image):
    d = ImageDraw.Draw(img)

    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 78, x, H - 64], fill=(*CYAN, 10), width=1)

    for y in range(100, H - 70, 60):
        d.line([70, y, W - 70, y], fill=(*CYAN, 9), width=1)


def hr_box():
    return 82, 128, 835, 525


def hr_map(log_temp: float, log_lum: float):
    x0, y0, x1, y1 = hr_box()

    t_min, t_max = 3.35, 4.65
    l_min, l_max = -4.8, 6.6

    x = x0 + (t_max - log_temp) / (t_max - t_min) * (x1 - x0)
    y = y1 - (log_lum - l_min) / (l_max - l_min) * (y1 - y0)

    return x, y


def spectral_color(log_temp: float):
    if log_temp > 4.45:
        return (110, 170, 255)
    if log_temp > 4.20:
        return (80, 220, 255)
    if log_temp > 3.98:
        return (180, 255, 255)
    if log_temp > 3.82:
        return (130, 255, 140)
    if log_temp > 3.68:
        return (255, 240, 60)
    if log_temp > 3.52:
        return (255, 150, 45)
    return (255, 60, 45)

def generate_hr_catalog(seed: int = 31415):
    rng = np.random.default_rng(seed)
    stars = []

    for _ in range(1450):
        u = rng.beta(1.25, 1.15)
        log_t = lerp(4.52, 3.42, u) + rng.normal(0, 0.045)
        log_l = lerp(5.3, -3.2, u) + rng.normal(0, 0.42)
        r = rng.uniform(0.7, 1.7)
        stars.append(("main", log_t, log_l, r))

    for _ in range(420):
        u = rng.random()
        log_t = lerp(3.72, 3.45, u) + rng.normal(0, 0.035)
        log_l = lerp(1.0, 4.1, u) + rng.normal(0, 0.35)
        r = rng.uniform(1.0, 2.2)
        stars.append(("giant", log_t, log_l, r))

    for _ in range(220):
        log_t = rng.uniform(3.55, 4.15)
        log_l = rng.normal(5.7, 0.28)
        r = rng.uniform(1.2, 2.6)
        stars.append(("supergiant", log_t, log_l, r))

    for _ in range(360):
        u = rng.random()
        log_t = lerp(4.42, 3.78, u) + rng.normal(0, 0.04)
        log_l = lerp(-1.0, -3.8, u) + rng.normal(0, 0.28)
        r = rng.uniform(0.6, 1.4)
        stars.append(("wd", log_t, log_l, r))

    rng.shuffle(stars)
    return stars


STAR_CATALOG = generate_hr_catalog()


def draw_hr_axes(layer: Image.Image, font_axis, font_tick):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = hr_box()

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 120))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 110), width=1)

    temp_ticks = [
        (4.60, "40000 K"),
        (4.30, "20000 K"),
        (4.00, "10000 K"),
        (3.70, "5000 K"),
        (3.40, "2500 K"),
    ]

    for log_t, label in temp_ticks:
        x, _ = hr_map(log_t, -4.8)
        d.line([x, y0, x, y1], fill=(*CYAN, 26), width=1)
        d.text((x - 36, y1 + 12), label, font=font_tick, fill=(*CYAN2, 185))

    lum_ticks = [
        (6, "10^6"),
        (4, "10^4"),
        (2, "10^2"),
        (0, "1"),
        (-2, "10^-2"),
        (-4, "10^-4"),
    ]

    for log_l, label in lum_ticks:
        _, y = hr_map(3.35, log_l)
        d.line([x0, y, x1, y], fill=(*CYAN, 24), width=1)
        d.text((x0 - 72, y - 9), label, font=font_tick, fill=(*CYAN2, 185))

    d.text((x0 + 8, y0 - 30), "Luminosity  L / Lsun", font=font_axis, fill=(*CYAN2, 215))
    d.text((x0 + 285, y1 + 44), "Temperature  T_eff, K", font=font_axis, fill=(*CYAN2, 215))


def draw_region_labels(layer: Image.Image, font_mid):
    d = ImageDraw.Draw(layer)
    d.text((236, 314), "Main Sequence", font=font_mid, fill=(*CYAN2, 150))
    d.text((650, 142), "Supergiants", font=font_mid, fill=(*CYAN2, 165))
    d.text((672, 292), "Giants", font=font_mid, fill=(*CYAN2, 150))
    d.text((286, 445), "White Dwarfs", font=font_mid, fill=(*CYAN2, 145))


def draw_star_cloud(layer: Image.Image):
    d = ImageDraw.Draw(layer)

    for idx, (_kind, log_t, log_l, radius) in enumerate(STAR_CATALOG):
        x, y = hr_map(log_t, log_l)
        color = spectral_color(log_t)
        alpha = 38 if idx % 7 else 72

        d.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            fill=(*color, alpha),
        )


def track_state(progress: float):
    age_myr = progress * TOTAL_MYR

    for stage in TRACK:
        if stage["t0"] <= age_myr <= stage["t1"]:
            local = (age_myr - stage["t0"]) / max(1e-9, stage["t1"] - stage["t0"])
            local_s = smoothstep(local)

            log_t = lerp(stage["logT0"], stage["logT1"], local_s)
            log_l = lerp(stage["logL0"], stage["logL1"], local_s)
            radius = lerp(stage["r0"], stage["r1"], local_s)
            color = mix_color(stage["c0"], stage["c1"], local_s)

            return age_myr, stage["label"], log_t, log_l, radius, color, local

    stage = TRACK[-1]
    return TOTAL_MYR, stage["label"], stage["logT1"], stage["logL1"], stage["r1"], stage["c1"], 1.0


def sampled_track_points(progress: float, n: int = 220):
    pts = []
    max_age_myr = progress * TOTAL_MYR

    for age_myr in np.linspace(0, max_age_myr, n):
        p = age_myr / TOTAL_MYR
        _, _label, log_t, log_l, radius, color, _local = track_state(p)
        x, y = hr_map(log_t, log_l)
        pts.append((x, y, radius, color))

    return pts


def draw_evolution_trail(layer: Image.Image, progress: float):
    trail = Image.new("RGBA", layer.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(trail)

    pts = sampled_track_points(progress, n=240)

    if len(pts) < 2:
        return

    for idx in range(1, len(pts)):
        x0, y0, _r0, _c0 = pts[idx - 1]
        x1, y1, _r1, c1 = pts[idx]

        alpha = int(40 + 185 * idx / len(pts))
        width = int(5 + 7 * idx / len(pts))
        d.line([x0, y0, x1, y1], fill=(*c1, alpha), width=width)

    glow = trail.filter(ImageFilter.GaussianBlur(6))
    layer.alpha_composite(glow)
    layer.alpha_composite(trail)


def draw_stage_timeline(layer: Image.Image, progress: float, font):
    d = ImageDraw.Draw(layer)

    x0, y0, x1, y1 = 110, 614, 835, 644
    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 155), outline=(*CYAN, 90), width=1)

    for stage in TRACK:
        xa = x0 + (stage["t0"] / TOTAL_MYR) * (x1 - x0)
        xb = x0 + (stage["t1"] / TOTAL_MYR) * (x1 - x0)
        col = stage["c1"]
        d.rectangle([xa, y0, xb, y1], fill=(*col, 120))
        d.line([xa, y0, xa, y1], fill=(*CYAN2, 75), width=1)

    px = x0 + progress * (x1 - x0)
    d.line([px, y0 - 8, px, y1 + 8], fill=(*WHITE, 230), width=2)

    d.text(
        (x0, y1 + 10),
        "Evolution timeline, Myr",
        font=font,
        fill=(*CYAN2, 185),
    )


def draw_star_metrics_panel(
    layer: Image.Image,
    box,
    rows,
    font,
):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 18), "STAR METRICS", font=font, fill=(*CYAN2, 190))

    y = y0 + 56

    for key, value in rows:
        d.text((x0 + 22, y), key, font=font, fill=(*CYAN2, 170))
        d.text((x0 + 122, y), value, font=font, fill=(*GREEN, 225))
        y += 28


def draw_star_evolution_panel(layer, box, stage_label, progress, font):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 16), "STAR EVOLUTION", font=font, fill=(*CYAN2, 190))

    wrapped_stage = "\n".join(wrap_lines(stage_label, 16))

    if "WHITE DWARF" in stage_label:
        remnant = "White Dwarf"
        source = "Degenerate core"

    elif "BLUE DWARF" in stage_label:
        remnant = "White Dwarf"
        source = "Late H fusion"

    elif "RED DWARF" in stage_label:
        remnant = "Blue Dwarf"
        source = "Hydrogen fusion"

    else:
        remnant = "Red Dwarf"
        source = "Cloud collapse"

    yy = y0 + 52

    d.text((x0 + 22, yy), "Stage:", font=font, fill=(*CYAN2, 170))

    d.multiline_text(
        (x0 + 98, yy),
        wrapped_stage,
        font=font,
        fill=(*GREEN, 225),
        spacing=3,
    )

    yy += 52

    d.text((x0 + 22, yy), "Source:", font=font, fill=(*CYAN2, 170))
    d.text((x0 + 98, yy), source, font=font, fill=(*GREEN, 225))

    yy += 32

    d.text((x0 + 22, yy), "Remnant:", font=font, fill=(*CYAN2, 170))
    d.text((x0 + 98, yy), remnant, font=font, fill=(*GREEN, 225))


def draw_track_marker(layer: Image.Image, x: float, y: float, phase: float):
    d = ImageDraw.Draw(layer)

    pulse = 0.6 + 0.4 * np.sin(phase * 5) ** 2
    r = 8
    ring = 24 + 4 * pulse

    d.ellipse(
        [x - ring, y - ring, x + ring, y + ring],
        outline=(*GREEN, int(150 + 80 * pulse)),
        width=2,
    )

    d.line([x - 34, y, x - 12, y], fill=(*GREEN, 190), width=1)
    d.line([x + 12, y, x + 34, y], fill=(*GREEN, 190), width=1)
    d.line([x, y - 34, x, y - 12], fill=(*GREEN, 190), width=1)
    d.line([x, y + 12, x, y + 34], fill=(*GREEN, 190), width=1)

    d.ellipse(
        [x - r, y - r, x + r, y + r],
        fill=(*WHITE, 230),
        outline=(*GREEN, 230),
        width=1,
    )


def draw_star_symbol(
    layer: Image.Image,
    x: float,
    y: float,
    radius: float,
    color,
    phase: float,
    label: str,
    local_stage: float,
):
    d = ImageDraw.Draw(layer)
    pulse = 0.82 + 0.18 * np.sin(phase * 5) ** 2

    if "SUPERNOVA" in label:
        shock_r = radius * (0.55 + local_stage * 1.9)

        for scale, alpha, width in [(1.0, 220, 4), (1.35, 150, 3), (1.8, 80, 2), (2.35, 35, 1)]:
            rr = shock_r * scale
            d.ellipse([x - rr, y - rr, x + rr, y + rr], outline=(255, 220, 120, alpha), width=width)

        core_r = radius * (0.18 + 0.18 * np.sin(phase * 18) ** 2)
        d.ellipse([x - core_r, y - core_r, x + core_r, y + core_r], fill=(255, 255, 255, 245), outline=(255, 240, 180, 220), width=2)
        return

    if "NEUTRON STAR" in label:
        beam_len = radius * 8
        angle = phase * 5.5
        dx = np.cos(angle) * beam_len
        dy = np.sin(angle) * beam_len

        d.line([x - dx, y - dy, x + dx, y + dy], fill=(*CYAN2, 170), width=2)
        d.line([x + dy * 0.35, y - dx * 0.35, x - dy * 0.35, y + dx * 0.35], fill=(*PURPLE, 120), width=1)

        for scale, alpha in [(3.0, 18), (1.8, 45)]:
            rr = radius * scale
            d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(*CYAN2, alpha))

        d.ellipse([x - radius, y - radius, x + radius, y + radius], fill=(220, 245, 255, 245), outline=(255, 255, 255, 230), width=2)
        return

    if "T-DWARF" in label:
        for scale, alpha in [(3.0, 18), (2.0, 32), (1.25, 70)]:
            rr = radius * scale
            d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(90, 70, 120, alpha))

        d.ellipse([x - radius, y - radius, x + radius, y + radius], fill=(85, 55, 90, 230), outline=(150, 120, 180, 130), width=1)
        return

    if "Y-DWARF" in label:
        for scale, alpha in [(3.2, 14), (2.1, 24), (1.4, 42)]:
            rr = radius * scale
            d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(80, 55, 110, alpha))

        d.ellipse([x - radius, y - radius, x + radius, y + radius], fill=(35, 28, 48, 245), outline=(140, 115, 180, 120), width=1)
        return

    if "PROTOSTAR" in label or "MOLECULAR" in label or "PROTO-BROWN" in label:
        rng = np.random.default_rng(42)
        cloud_r = radius * 2.4

        for _ in range(32):
            a = rng.uniform(0, 2 * np.pi)
            rr = rng.uniform(0.1, 1.0) * cloud_r
            px = x + np.cos(a) * rr * 0.9
            py = y + np.sin(a) * rr * 0.6
            pr = rng.uniform(radius * 0.25, radius * 0.8)

            d.ellipse([px - pr, py - pr, px + pr, py + pr], fill=(255, 120, 60, int(rng.uniform(16, 48))))

    if "RED SUPERGIANT" in label:
        shell_r = radius * (1.35 + 0.06 * np.sin(phase * 4))
        d.ellipse([x - shell_r, y - shell_r, x + shell_r, y + shell_r], fill=(*RED, 32), outline=(*ORANGE, 90), width=2)

    if "CORE COLLAPSE" in label:
        shock_r = radius * (1.15 + 0.65 * local_stage)
        d.ellipse([x - shock_r, y - shock_r, x + shock_r, y + shock_r], outline=(*WHITE, int(220 * (1.0 - local_stage))), width=3)

    if "RED DWARF" in label:
        flare = 0.5 + 0.5 * np.sin(phase * 14) ** 2
        for scale, alpha in [(2.6, 14), (1.8, 36), (1.2, 72)]:
            rr = radius * scale
            d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(255, 80, 60, int(alpha * flare)))

    if "BLUE DWARF" in label:
        for scale, alpha in [(2.4, 20), (1.7, 52), (1.2, 88)]:
            rr = radius * scale
            d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(160, 220, 255, alpha))

    if "L-DWARF" in label:
        for scale, alpha in [(2.8, 16), (1.9, 34), (1.3, 70)]:
            rr = radius * scale
            d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(180, 80, 55, alpha))

    for scale, alpha in [(2.6, 22), (1.8, 48), (1.25, 85)]:
        rr = radius * scale * pulse
        d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(*color, alpha))

    d.ellipse([x - radius, y - radius, x + radius, y + radius], fill=(*color, 235), outline=(*WHITE, 220), width=2)


def draw_empty_star_preview_panel(
    layer,
    box,
    font,
    radius,
    color,
    phase,
    stage_label,
    local_stage,
):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 18), "STAR VIEW", font=font, fill=(*CYAN2, 190))

    cx = (x0 + x1) / 2
    cy = y0 + 108

    panel_w = x1 - x0
    panel_h = y1 - y0
    max_preview_radius = min(panel_w * 0.18, panel_h * 0.16)

    if "MOLECULAR" in stage_label:
        scaled_radius = max_preview_radius * 0.85
    elif "PROTO-BROWN" in stage_label:
        scaled_radius = max_preview_radius * 0.72
    elif "L-DWARF" in stage_label:
        scaled_radius = max_preview_radius * 0.55
    elif "T-DWARF" in stage_label:
        scaled_radius = max_preview_radius * 0.46
    elif "Y-DWARF" in stage_label:
        scaled_radius = max_preview_radius * 0.42
    else:
        scaled_radius = max_preview_radius * 0.50

    draw_star_symbol(
        layer=layer,
        x=cx,
        y=cy,
        radius=scaled_radius,
        color=color,
        phase=phase,
        label=stage_label,
        local_stage=local_stage,
    )


def draw_stage_popup(layer, x, y, stage_label, font):
    d = ImageDraw.Draw(layer)

    label_lines = wrap_lines(stage_label, 18)

    box_x0 = x + 32
    box_y0 = y - 22
    box_x1 = box_x0 + 185
    box_y1 = box_y0 + 24 + 18 * len(label_lines)

    if box_x1 > 840:
        box_x0 = x - 220
        box_x1 = box_x0 + 185

    if box_y0 < 90:
        box_y0 = y + 24
        box_y1 = box_y0 + 24 + 18 * len(label_lines)

    d.rectangle(
        [box_x0, box_y0, box_x1, box_y1],
        fill=(*BG_DARK, 210),
        outline=(*GREEN, 180),
        width=1,
    )

    yy = box_y0 + 10

    for line in label_lines:
        d.text((box_x0 + 10, yy), line, font=font, fill=(*GREEN, 230))
        yy += 18


def draw_hr_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)

    font_big = load_font(20)
    font_mid = load_font(19)
    font_small = load_font(15)
    font_tick = load_font(15)
    font_axis = load_font(17)

    phase = 2 * np.pi * i / TOTAL_FRAMES
    progress = visual_progress_with_stage_holds(i)

    age_myr, stage_label, log_t, log_l, radius, color, local_stage = track_state(progress)

    draw_hud_frame(frame, font_big)
    draw_grid_background(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))

    draw_hr_axes(scene, font_axis, font_tick)
    draw_star_cloud(scene)
    draw_region_labels(scene, font_mid)

    draw_evolution_trail(scene, progress)

    x, y = hr_map(log_t, log_l)
    draw_track_marker(scene, x, y, phase)

    if is_visual_hold_frame(i):
        draw_stage_popup(scene, x, y, stage_label, font_small)

    draw_stage_timeline(scene, progress, font_small)

    temp = 10 ** log_t
    lum = 10 ** log_l

    metrics_rows = [
        ("mass", "0.12 Msun"),
        ("age", f"{age_myr:.0f} Gyr"),
        ("T_eff", f"{temp:,.0f} K"),
        ("log L", f"{log_l:+.2f}"),
        ("radius", f"{max(0.01, radius * 0.06):.2f} Rsun"),
        ("fate", "white dwarf"),
    ]

    draw_empty_star_preview_panel(
        scene,
        (900, 53, 1155, 256),
        font_small,
        radius,
        color,
        phase,
        stage_label,
        local_stage,
    )

    draw_star_evolution_panel(scene, (900, 265, 1155, 435), stage_label, progress, font_small)
    draw_star_metrics_panel(scene, (900, 450, 1155, 670), metrics_rows, font_small)

    glow_composite(frame, scene, blur=4)
    return frame


def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False):
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "34",
            "-pix_fmt", "yuva420p",
            "-auto-alt-ref", "0",
            "-row-mt", "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False):
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "22",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str):
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] stellar evolution track: heavy star")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")
        frames.append(draw_hr_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(f"Created 1 animation(s) as '{output_format}' in '{ANIMATIONS_DIR.resolve()}'")


main()

[START] stellar evolution track: heavy star
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 456
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/456
[GENERATE] frame 24/456
[GENERATE] frame 48/456
[GENERATE] frame 72/456
[GENERATE] frame 96/456
[GENERATE] frame 120/456
[GENERATE] frame 144/456
[GENERATE] frame 168/456
[GENERATE] frame 192/456
[GENERATE] frame 216/456
[GENERATE] frame 240/456
[GENERATE] frame 264/456
[GENERATE] frame 288/456
[GENERATE] frame 312/456
[GENERATE] frame 336/456
[GENERATE] frame 360/456
[GENERATE] frame 384/456
[GENERATE] frame 408/456
[GENERATE] frame 432/456


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] stellar_evolution_red_dwarf
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/stellar_evolution_red_dwarf/stellar_evolution_red_dwarf.webm

Created 1 animation(s) as 'webm' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


[out#0/webm @ 0x130622cf0] video:555KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 62.155489%
frame=  456 fps= 37 q=34.0 Lsize=     900KiB time=00:00:19.00 bitrate= 388.2kbits/s speed=1.54x    


# Brown Dwarf

In [47]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import textwrap
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


OUTPUT_FORMAT = "webm"
FPS = 24

TOTAL_GYR = 100.0

TRACK = [

    {
        "label": "MOLECULAR CLOUD CORE",
        "t0": 0.0,
        "t1": 0.08,

        "logT0": 3.35,
        "logL0": -0.4,

        "logT1": 3.25,
        "logL1": -1.2,

        "r0": 46,
        "r1": 32,

        "c0": (255, 140, 90),
        "c1": (255, 170, 120),
    },

    {
        "label": "PROTO-BROWN DWARF",
        "t0": 0.08,
        "t1": 0.5,

        "logT0": 3.25,
        "logL0": -1.2,

        "logT1": 3.18,
        "logL1": -2.0,

        "r0": 32,
        "r1": 22,

        "c0": (255, 170, 120),
        "c1": (255, 120, 70),
    },

    {
        "label": "L-DWARF",
        "t0": 0.5,
        "t1": 5.0,

        "logT0": 3.18,
        "logL0": -2.0,

        "logT1": 3.02,
        "logL1": -3.3,

        "r0": 22,
        "r1": 14,

        "c0": (255, 120, 70),
        "c1": (210, 90, 70),
    },

    {
        "label": "T-DWARF",
        "t0": 5.0,
        "t1": 25.0,

        "logT0": 3.02,
        "logL0": -3.3,

        "logT1": 2.75,
        "logL1": -4.9,

        "r0": 14,
        "r1": 10,

        "c0": (210, 90, 70),
        "c1": (120, 80, 110),
    },

    {
        "label": "Y-DWARF",
        "t0": 25.0,
        "t1": 100.0,

        "logT0": 2.75,
        "logL0": -4.9,

        "logT1": 2.45,
        "logL1": -6.0,

        "r0": 10,
        "r1": 7,

        "c0": (120, 80, 110),
        "c1": (40, 35, 55),
    },

]

ANIMATION_NAME = "stellar_evolution_brown_dwarf"
ANIMATIONS_DIR = Path("media-site/animations")

CANVAS_SIZE = (1200, 720)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)

MOVE_FRAMES = 42
HOLD_FRAMES = FPS * 3


def smoothstep(t: float) -> float:
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3 - 2 * t)


def lerp(a, b, t):
    return a + (b - a) * t


def mix_color(c1, c2, t):
    t = float(np.clip(t, 0, 1))
    return tuple(int(lerp(a, b, t)) for a, b in zip(c1, c2))


def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 4):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def build_visual_timeline():
    segments = []

    for stage in TRACK:
        p0 = stage["t0"] / TOTAL_MYR
        p1 = stage["t1"] / TOTAL_MYR

        segments.append({
            "type": "move",
            "p0": p0,
            "p1": p1,
            "frames": MOVE_FRAMES,
        })

        segments.append({
            "type": "hold",
            "p0": p1,
            "p1": p1,
            "frames": HOLD_FRAMES,
        })

    return segments


VISUAL_TIMELINE = build_visual_timeline()
TOTAL_FRAMES = sum(s["frames"] for s in VISUAL_TIMELINE)


def visual_progress_with_stage_holds(frame_index: int) -> float:
    frame_index %= TOTAL_FRAMES
    cursor = 0

    for segment in VISUAL_TIMELINE:
        start = cursor
        end = cursor + segment["frames"]

        if start <= frame_index < end:

            if segment["type"] == "hold":
                return segment["p0"]

            local = (frame_index - start) / max(1, segment["frames"] - 1)
            local = smoothstep(local)

            return lerp(segment["p0"], segment["p1"], local)

        cursor = end

    return 1.0


def is_visual_hold_frame(frame_index: int) -> bool:
    frame_index %= TOTAL_FRAMES
    cursor = 0

    for segment in VISUAL_TIMELINE:
        start = cursor
        end = cursor + segment["frames"]

        if start <= frame_index < end:
            return segment["type"] == "hold"

        cursor = end

    return False


def draw_cut_panel(
    d: ImageDraw.ImageDraw,
    box,
    fill=(*BG_DARK, 165),
    outline=(*CYAN, 105),
    cut: int = 18,
):
    x0, y0, x1, y1 = box

    pts = [
        (x0 + cut, y0),
        (x1, y0),
        (x1, y1 - cut),
        (x1 - cut, y1),
        (x0, y1),
        (x0, y0 + cut),
    ]

    d.polygon(pts, fill=fill)
    d.line(pts + [pts[0]], fill=outline, width=1)


def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)

    W, H = img.size
    pad = 28
    cut = 48

    pts = [
        (pad + cut, pad),
        (W - pad - 260, pad),
        (W - pad - 230, pad + 18),
        (W - pad, pad + 18),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)

    d.rectangle(
        [pad + 18, pad + 38, W - pad - 18, H - pad - 44],
        outline=(*CYAN2, 38),
        width=1,
    )

    d.text(
        (pad + 24, pad + 8),
        "TELEMETRY // STELLAR EVOLUTION TRACK // BROWN DWARF (0.08 Msun)",
        font=font,
        fill=(*CYAN2, 225),
    )


def draw_grid_background(img: Image.Image):
    d = ImageDraw.Draw(img)

    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 78, x, H - 64], fill=(*CYAN, 10), width=1)

    for y in range(100, H - 70, 60):
        d.line([70, y, W - 70, y], fill=(*CYAN, 9), width=1)


def hr_box():
    return 82, 128, 835, 525


def hr_map(log_temp: float, log_lum: float):
    x0, y0, x1, y1 = hr_box()

    t_min, t_max = 2.35, 4.65
    l_min, l_max = -6.4, 6.6

    x = x0 + (t_max - log_temp) / (t_max - t_min) * (x1 - x0)
    y = y1 - (log_lum - l_min) / (l_max - l_min) * (y1 - y0)

    return x, y


def spectral_color(log_temp: float):
    if log_temp > 4.45:
        return (110, 170, 255)
    if log_temp > 4.20:
        return (80, 220, 255)
    if log_temp > 3.98:
        return (180, 255, 255)
    if log_temp > 3.82:
        return (130, 255, 140)
    if log_temp > 3.68:
        return (255, 240, 60)
    if log_temp > 3.52:
        return (255, 150, 45)
    return (255, 60, 45)

def generate_hr_catalog(seed: int = 31415):
    rng = np.random.default_rng(seed)
    stars = []

    for _ in range(1450):
        u = rng.beta(1.25, 1.15)
        log_t = lerp(4.52, 3.42, u) + rng.normal(0, 0.045)
        log_l = lerp(5.3, -3.2, u) + rng.normal(0, 0.42)
        r = rng.uniform(0.7, 1.7)
        stars.append(("main", log_t, log_l, r))

    for _ in range(420):
        u = rng.random()
        log_t = lerp(3.72, 3.45, u) + rng.normal(0, 0.035)
        log_l = lerp(1.0, 4.1, u) + rng.normal(0, 0.35)
        r = rng.uniform(1.0, 2.2)
        stars.append(("giant", log_t, log_l, r))

    for _ in range(220):
        log_t = rng.uniform(3.55, 4.15)
        log_l = rng.normal(5.7, 0.28)
        r = rng.uniform(1.2, 2.6)
        stars.append(("supergiant", log_t, log_l, r))

    for _ in range(360):
        u = rng.random()
        log_t = lerp(4.42, 3.78, u) + rng.normal(0, 0.04)
        log_l = lerp(-1.0, -3.8, u) + rng.normal(0, 0.28)
        r = rng.uniform(0.6, 1.4)
        stars.append(("wd", log_t, log_l, r))

    rng.shuffle(stars)
    return stars


STAR_CATALOG = generate_hr_catalog()


def draw_hr_axes(layer: Image.Image, font_axis, font_tick):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = hr_box()

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 120))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 110), width=1)

    temp_ticks = [
        (4.60, "40000 K"),
        (4.30, "20000 K"),
        (4.00, "10000 K"),
        (3.70, "5000 K"),
        (3.40, "2500 K"),
        (3.00, "1000 K"),
        (2.70, "500 K"),
        (2.45, "300 K"),
    ]

    for log_t, label in temp_ticks:
        x, _ = hr_map(log_t, -4.8)
        d.line([x, y0, x, y1], fill=(*CYAN, 26), width=1)
        d.text((x - 30, y1 + 16), label, font=font_tick, fill=(*CYAN2, 185))
        
    lum_ticks = [
        (6, "10^6"),
        (4, "10^4"),
        (2, "10^2"),
        (0, "1"),
        (-2, "10^-2"),
        (-4, "10^-4"),
    ]

    for log_l, label in lum_ticks:
        _, y = hr_map(3.35, log_l)
        d.line([x0, y, x1, y], fill=(*CYAN, 24), width=1)
        d.text((x0 - 72, y - 9), label, font=font_tick, fill=(*CYAN2, 185))

    d.text((x0 + 8, y0 - 30), "Luminosity  L / Lsun", font=font_axis, fill=(*CYAN2, 215))
    d.text((x0 + 285, y1 + 44), "Temperature  T_eff, K", font=font_axis, fill=(*CYAN2, 215))


def draw_region_labels(layer: Image.Image, font_mid):
    d = ImageDraw.Draw(layer)
    d.text((236, 314), "Main Sequence", font=font_mid, fill=(*CYAN2, 150))
    d.text((650, 142), "Supergiants", font=font_mid, fill=(*CYAN2, 165))
    d.text((672, 292), "Giants", font=font_mid, fill=(*CYAN2, 150))
    d.text((286, 445), "White Dwarfs", font=font_mid, fill=(*CYAN2, 145))


def draw_star_cloud(layer: Image.Image):
    d = ImageDraw.Draw(layer)

    for idx, (_kind, log_t, log_l, radius) in enumerate(STAR_CATALOG):
        x, y = hr_map(log_t, log_l)
        color = spectral_color(log_t)
        alpha = 38 if idx % 7 else 72

        d.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            fill=(*color, alpha),
        )


def track_state(progress: float):
    age_myr = progress * TOTAL_MYR

    for stage in TRACK:
        if stage["t0"] <= age_myr <= stage["t1"]:
            local = (age_myr - stage["t0"]) / max(1e-9, stage["t1"] - stage["t0"])
            local_s = smoothstep(local)

            log_t = lerp(stage["logT0"], stage["logT1"], local_s)
            log_l = lerp(stage["logL0"], stage["logL1"], local_s)
            radius = lerp(stage["r0"], stage["r1"], local_s)
            color = mix_color(stage["c0"], stage["c1"], local_s)

            return age_myr, stage["label"], log_t, log_l, radius, color, local

    stage = TRACK[-1]
    return TOTAL_MYR, stage["label"], stage["logT1"], stage["logL1"], stage["r1"], stage["c1"], 1.0


def sampled_track_points(progress: float, n: int = 220):
    pts = []
    max_age_myr = progress * TOTAL_MYR

    for age_myr in np.linspace(0, max_age_myr, n):
        p = age_myr / TOTAL_MYR
        _, _label, log_t, log_l, radius, color, _local = track_state(p)
        x, y = hr_map(log_t, log_l)
        pts.append((x, y, radius, color))

    return pts


def draw_evolution_trail(layer: Image.Image, progress: float):
    trail = Image.new("RGBA", layer.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(trail)

    pts = sampled_track_points(progress, n=240)

    if len(pts) < 2:
        return

    for idx in range(1, len(pts)):
        x0, y0, _r0, _c0 = pts[idx - 1]
        x1, y1, _r1, c1 = pts[idx]

        alpha = int(40 + 185 * idx / len(pts))
        width = int(5 + 7 * idx / len(pts))
        d.line([x0, y0, x1, y1], fill=(*c1, alpha), width=width)

    glow = trail.filter(ImageFilter.GaussianBlur(6))
    layer.alpha_composite(glow)
    layer.alpha_composite(trail)


def draw_stage_timeline(layer: Image.Image, progress: float, font):
    d = ImageDraw.Draw(layer)

    x0, y0, x1, y1 = 110, 614, 835, 644
    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 155), outline=(*CYAN, 90), width=1)

    for stage in TRACK:
        xa = x0 + (stage["t0"] / TOTAL_MYR) * (x1 - x0)
        xb = x0 + (stage["t1"] / TOTAL_MYR) * (x1 - x0)
        col = stage["c1"]
        d.rectangle([xa, y0, xb, y1], fill=(*col, 120))
        d.line([xa, y0, xa, y1], fill=(*CYAN2, 75), width=1)

    px = x0 + progress * (x1 - x0)
    d.line([px, y0 - 8, px, y1 + 8], fill=(*WHITE, 230), width=2)

    d.text(
        (x0, y1 + 10),
        "Evolution timeline, Myr",
        font=font,
        fill=(*CYAN2, 185),
    )


def draw_star_metrics_panel(
    layer: Image.Image,
    box,
    rows,
    font,
):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 18), "STAR METRICS", font=font, fill=(*CYAN2, 190))

    y = y0 + 56

    for key, value in rows:
        d.text((x0 + 22, y), key, font=font, fill=(*CYAN2, 170))
        d.text((x0 + 122, y), value, font=font, fill=(*GREEN, 225))
        y += 28


def draw_star_evolution_panel(layer, box, stage_label, progress, font):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 16), "STAR EVOLUTION", font=font, fill=(*CYAN2, 190))

    wrapped_stage = "\n".join(wrap_lines(stage_label, 16))

    if "WHITE DWARF" in stage_label:
        remnant = "White Dwarf"
        source = "Degenerate core"

    elif "BLUE DWARF" in stage_label:
        remnant = "White Dwarf"
        source = "Late H fusion"

    elif "RED DWARF" in stage_label:
        remnant = "Blue Dwarf"
        source = "Hydrogen fusion"

    else:
        remnant = "Red Dwarf"
        source = "Cloud collapse"

    yy = y0 + 52

    d.text((x0 + 22, yy), "Stage:", font=font, fill=(*CYAN2, 170))

    d.multiline_text(
        (x0 + 98, yy),
        wrapped_stage,
        font=font,
        fill=(*GREEN, 225),
        spacing=3,
    )

    yy += 52

    d.text((x0 + 22, yy), "Source:", font=font, fill=(*CYAN2, 170))
    d.text((x0 + 98, yy), source, font=font, fill=(*GREEN, 225))

    yy += 32

    d.text((x0 + 22, yy), "Remnant:", font=font, fill=(*CYAN2, 170))
    d.text((x0 + 98, yy), remnant, font=font, fill=(*GREEN, 225))


def draw_track_marker(layer: Image.Image, x: float, y: float, phase: float):
    d = ImageDraw.Draw(layer)

    pulse = 0.6 + 0.4 * np.sin(phase * 5) ** 2
    r = 8
    ring = 24 + 4 * pulse

    d.ellipse(
        [x - ring, y - ring, x + ring, y + ring],
        outline=(*GREEN, int(150 + 80 * pulse)),
        width=2,
    )

    d.line([x - 34, y, x - 12, y], fill=(*GREEN, 190), width=1)
    d.line([x + 12, y, x + 34, y], fill=(*GREEN, 190), width=1)
    d.line([x, y - 34, x, y - 12], fill=(*GREEN, 190), width=1)
    d.line([x, y + 12, x, y + 34], fill=(*GREEN, 190), width=1)

    d.ellipse(
        [x - r, y - r, x + r, y + r],
        fill=(*WHITE, 230),
        outline=(*GREEN, 230),
        width=1,
    )


def draw_star_symbol(
    layer: Image.Image,
    x: float,
    y: float,
    radius: float,
    color,
    phase: float,
    label: str,
    local_stage: float,
):
    d = ImageDraw.Draw(layer)
    pulse = 0.82 + 0.18 * np.sin(phase * 5) ** 2

    if "SUPERNOVA" in label:
        shock_r = radius * (0.55 + local_stage * 1.9)

        for scale, alpha, width in [
            (1.0, 220, 4),
            (1.35, 150, 3),
            (1.8, 80, 2),
            (2.35, 35, 1),
        ]:
            rr = shock_r * scale
            d.ellipse(
                [x - rr, y - rr, x + rr, y + rr],
                outline=(255, 220, 120, alpha),
                width=width,
            )

        core_r = radius * (0.18 + 0.18 * np.sin(phase * 18) ** 2)
        d.ellipse(
            [x - core_r, y - core_r, x + core_r, y + core_r],
            fill=(255, 255, 255, 245),
            outline=(255, 240, 180, 220),
            width=2,
        )
        return

    if "NEUTRON STAR" in label:
        beam_len = radius * 8
        angle = phase * 5.5

        dx = np.cos(angle) * beam_len
        dy = np.sin(angle) * beam_len

        d.line([x - dx, y - dy, x + dx, y + dy], fill=(*CYAN2, 170), width=2)
        d.line(
            [x + dy * 0.35, y - dx * 0.35, x - dy * 0.35, y + dx * 0.35],
            fill=(*PURPLE, 120),
            width=1,
        )

        for scale, alpha in [(3.0, 18), (1.8, 45)]:
            rr = radius * scale
            d.ellipse([x - rr, y - rr, x + rr, y + rr], fill=(*CYAN2, alpha))

        d.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            fill=(220, 245, 255, 245),
            outline=(255, 255, 255, 230),
            width=2,
        )
        return

    if "PROTOSTAR" in label:
        rng = np.random.default_rng(42)
        cloud_r = radius * 2.4

        for _ in range(32):
            a = rng.uniform(0, 2 * np.pi)
            rr = rng.uniform(0.1, 1.0) * cloud_r
            px = x + np.cos(a) * rr * 0.9
            py = y + np.sin(a) * rr * 0.6
            pr = rng.uniform(radius * 0.25, radius * 0.8)

            d.ellipse(
                [px - pr, py - pr, px + pr, py + pr],
                fill=(255, 120, 60, int(rng.uniform(16, 48))),
            )

    if "RED SUPERGIANT" in label:
        shell_r = radius * (1.35 + 0.06 * np.sin(phase * 4))
        d.ellipse(
            [x - shell_r, y - shell_r, x + shell_r, y + shell_r],
            fill=(*RED, 32),
            outline=(*ORANGE, 90),
            width=2,
        )

    if "CORE COLLAPSE" in label:
        shock_r = radius * (1.15 + 0.65 * local_stage)
        d.ellipse(
            [x - shock_r, y - shock_r, x + shock_r, y + shock_r],
            outline=(*WHITE, int(220 * (1.0 - local_stage))),
            width=3,
        )

    if "RED DWARF" in label:

        flare = 0.5 + 0.5 * np.sin(phase * 14) ** 2

        for scale, alpha in [
            (2.6, 14),
            (1.8, 36),
            (1.2, 72),
        ]:
            rr = radius * scale
            d.ellipse(
                [x - rr, y - rr, x + rr, y + rr],
                fill=((255), 80, 60, int(alpha * flare)),
            )

    if "BLUE DWARF" in label:

        for scale, alpha in [
            (2.4, 20),
            (1.7, 52),
            (1.2, 88),
        ]:
            rr = radius * scale
            d.ellipse(
                [x - rr, y - rr, x + rr, y + rr],
                fill=(160, 220, 255, alpha),
            )

    if "Y-DWARF" in label:

        for scale, alpha in [
            (3.0, 6),
            (2.0, 12),
            (1.4, 18),
        ]:
            rr = radius * scale

            d.ellipse(
                [x - rr, y - rr, x + rr, y + rr],
                fill=(60, 40, 80, alpha),
            )

        d.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            fill=(25, 20, 35, 255),
            outline=(110, 90, 140, 90),
            width=1,
        )

        return



def draw_empty_star_preview_panel(
    layer,
    box,
    font,
    radius,
    color,
    phase,
    stage_label,
    local_stage,
):
    d = ImageDraw.Draw(layer)
    draw_cut_panel(d, box)

    x0, y0, x1, y1 = box

    d.text((x0 + 22, y0 + 18), "STAR VIEW", font=font, fill=(*CYAN2, 190))

    cx = (x0 + x1) / 2
    cy = y0 + 108

    panel_w = x1 - x0
    panel_h = y1 - y0
    max_preview_radius = min(panel_w * 0.18, panel_h * 0.16)

    if "SUPERNOVA" in stage_label:
        scaled_radius = max_preview_radius * 0.72
    elif "NEUTRON STAR" in stage_label:
        scaled_radius = max_preview_radius * 0.22
    elif "CORE COLLAPSE" in stage_label:
        scaled_radius = max_preview_radius * 0.62
    elif "RED SUPERGIANT" in stage_label:
        scaled_radius = max_preview_radius * 0.95
    elif "PROTOSTAR" in stage_label:
        scaled_radius = max_preview_radius * 0.68
    else:
        scaled_radius = min(radius * 0.70, max_preview_radius)

    draw_star_symbol(
        layer=layer,
        x=cx,
        y=cy,
        radius=scaled_radius,
        color=color,
        phase=phase,
        label=stage_label,
        local_stage=local_stage,
    )


def draw_stage_popup(layer, x, y, stage_label, font):
    d = ImageDraw.Draw(layer)

    label_lines = wrap_lines(stage_label, 18)

    box_x0 = x + 32
    box_y0 = y - 22
    box_x1 = box_x0 + 185
    box_y1 = box_y0 + 24 + 18 * len(label_lines)

    if box_x1 > 840:
        box_x0 = x - 220
        box_x1 = box_x0 + 185

    if box_y0 < 90:
        box_y0 = y + 24
        box_y1 = box_y0 + 24 + 18 * len(label_lines)

    d.rectangle(
        [box_x0, box_y0, box_x1, box_y1],
        fill=(*BG_DARK, 210),
        outline=(*GREEN, 180),
        width=1,
    )

    yy = box_y0 + 10

    for line in label_lines:
        d.text((box_x0 + 10, yy), line, font=font, fill=(*GREEN, 230))
        yy += 18


def draw_hr_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)

    font_big = load_font(20)
    font_mid = load_font(19)
    font_small = load_font(15)
    font_tick = load_font(15)
    font_axis = load_font(17)

    phase = 2 * np.pi * i / TOTAL_FRAMES
    progress = visual_progress_with_stage_holds(i)

    age_myr, stage_label, log_t, log_l, radius, color, local_stage = track_state(progress)

    draw_hud_frame(frame, font_big)
    draw_grid_background(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))

    draw_hr_axes(scene, font_axis, font_tick)
    draw_star_cloud(scene)
    draw_region_labels(scene, font_mid)

    draw_evolution_trail(scene, progress)

    x, y = hr_map(log_t, log_l)
    draw_track_marker(scene, x, y, phase)

    if is_visual_hold_frame(i):
        draw_stage_popup(scene, x, y, stage_label, font_small)

    draw_stage_timeline(scene, progress, font_small)

    temp = 10 ** log_t
    lum = 10 ** log_l

    metrics_rows = [
        ("mass", "0.12 Msun"),
        ("age", f"{age_myr:.0f} Gyr"),
        ("T_eff", f"{temp:,.0f} K"),
        ("log L", f"{log_l:+.2f}"),
        ("radius", f"{max(0.01, radius * 0.06):.2f} Rsun"),
        ("fate", "white dwarf"),
    ]

    draw_empty_star_preview_panel(
        scene,
        (900, 53, 1155, 256),
        font_small,
        radius,
        color,
        phase,
        stage_label,
        local_stage,
    )

    draw_star_evolution_panel(scene, (900, 265, 1155, 435), stage_label, progress, font_small)
    draw_star_metrics_panel(scene, (900, 450, 1155, 670), metrics_rows, font_small)

    glow_composite(frame, scene, blur=4)
    return frame


def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False):
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "34",
            "-pix_fmt", "yuva420p",
            "-auto-alt-ref", "0",
            "-row-mt", "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False):
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "22",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str):
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] stellar evolution track: heavy star")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")
        frames.append(draw_hr_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(f"Created 1 animation(s) as '{output_format}' in '{ANIMATIONS_DIR.resolve()}'")


main()

[START] stellar evolution track: heavy star
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 570
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/570
[GENERATE] frame 24/570
[GENERATE] frame 48/570
[GENERATE] frame 72/570
[GENERATE] frame 96/570
[GENERATE] frame 120/570
[GENERATE] frame 144/570
[GENERATE] frame 168/570
[GENERATE] frame 192/570
[GENERATE] frame 216/570
[GENERATE] frame 240/570
[GENERATE] frame 264/570
[GENERATE] frame 288/570
[GENERATE] frame 312/570
[GENERATE] frame 336/570
[GENERATE] frame 360/570
[GENERATE] frame 384/570
[GENERATE] frame 408/570
[GENERATE] frame 432/570
[GENERATE] frame 456/570
[GENERATE] frame 480/570
[GENERATE] frame 504/570
[GENERATE] frame 528/570
[GENERATE] frame 552/570


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] stellar_evolution_brown_dwarf
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/stellar_evolution_brown_dwarf/stellar_evolution_brown_dwarf.webm

Created 1 animation(s) as 'webm' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


[out#0/webm @ 0x11de25800] video:517KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 60.436375%
frame=  570 fps= 40 q=34.0 Lsize=     830KiB time=00:00:23.75 bitrate= 286.3kbits/s speed=1.69x    
